<a href="https://colab.research.google.com/github/GunjanBhandari-pixel/Gunnu/blob/main/Multiterm_2_term.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
#from scipy.linalg import solve
from datetime import datetime
start = datetime.now()
from scipy.linalg import solve_banded
from mpmath import mp
#from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import ScalarFormatter

mp.dps = 120;

a = mp.mpf("0.80")
a_s = mp.mpf("0.10")

g1 = mp.gamma(2 - a)
g1_s = mp.gamma(2 - a_s)
g = mp.gamma(3 - a)
g_s = mp.gamma(3 - a_s)

def f(x, t):
    x = mp.mpf(x)
    t = mp.mpf(t)
    return (mp.power(t,2-a)*(1/g)+mp.power(t,2-a_s)*(1/g_s))*2*mp.sin(2*mp.pi*x)+4*(mp.pi**2)*(t**2*mp.sin(2*mp.pi*x))

a_ = mp.mpf('1')
c_ = mp.mpf('0')
L  = mp.mpf('1')
T  = mp.mpf('1')

M=100
#N=math.floor(T*((M/L)**(6/(3-(a+a_s)/2))))
N=100
h = mp.mpf(L) / mp.mpf(M)
#r=mp.mpf(3/a)-1
r=mp.mpf(1)

#t=[T*((i/N)**r) for i in range(0,N+1)]
#t = T * (np.linspace(0, 1, N+1) ** r)
t = [T * (mp.mpf(i) / N) ** r for i in range(N + 1)]

#N1=math.ceil((mp.power(N,a-3)/t[1]**(2-a))**(1/(a-2)))
N1 = int(mp.ceil(
    mp.power(
        mp.power(N, a - 3) / mp.power(t[1], 2 - a),
        1 / (a - 2)
    )
))

#tF=[t[1]*((i/N1)**r) for i in range(0,N1+1)]
#tF = t[1] * np.linspace(0, 1, N1+1) ** r
tF = [t[1] * mp.power(mp.mpf(i) / N1, r) for i in range(N1 + 1)]

# Tau=np.zeros(N+1)
# for i in range(1,N+1):
#     Tau[i]=t[i]-t[i-1]
#Tau = np.diff(t, prepend=0)  # prepend=0  t[−1] = 0
Tau = [mp.mpf('0')] + [t[i] - t[i-1] for i in range(1, len(t))]

#x=[i*h for i in range(0,M+1)]
# x = np.linspace(0, L, M+1)
# X=np.zeros(M+3)

x = [mp.mpf(i) * L / M for i in range(M+1)]
X = [mp.mpf('0') for _ in range(M+3)]

X[0]=x[0]
X[1]=(x[0]+x[1])/2
for j in range(2,M+1):
    X[j]=x[j-1]
X[M+1]=(x[M]+x[M-1])/2
X[M+2]=x[M]

# u=np.zeros((M+3,N1+1))
# U=np.zeros((M+3,N+1))
# exact=np.zeros((M+3,N+1))
# error=np.zeros((M+3,N+1))

# A1 = np.zeros((M+5, M+5))
# B1=np.zeros((M+5,M+5))
# B=np.zeros((M+5,M+5))
# C=np.zeros((N+1,M+5))
# C1=np.zeros((N+1,M+5))
# C2=np.zeros((N+1,M+5))
# W1=np.zeros(M+5)
# W=np.zeros(M+5)

u     = mp.matrix(M+3, N1+1)
U     = mp.matrix(M+3, N+1)
exact = mp.matrix(M+3, N+1)
error = mp.matrix(M+3, N+1)

A1 = mp.matrix(M+5, M+5)
B1 = mp.matrix(M+5, M+5)
B  = mp.matrix(M+5, M+5)

C  = mp.matrix(N+1, M+5)
C1 = mp.matrix(N+1, M+5)
C2 = mp.matrix(N+1, M+5)

W1 = mp.matrix(M+5, 1)
W  = mp.matrix(M+5, 1)

zeta   = mp.matrix(N1+1, N1+1)
zeta_s = mp.matrix(N1+1, N1+1)

c1 = mp.mpf(69120) * h**2 / mp.mpf(a_)
d1 = mp.mpf(4320) * h**2 / mp.mpf(a_)

# --- Compute zeta ---
# for n in range(1, N1 + 1):
#     dt = tF[n] - tF[:n]
#     dt1 = tF[n] - tF[1:n + 1]
#     zeta[:n, n] = (dt**(1 - a) - dt1**(1 - a)) / ((tF[1:n + 1] - tF[:n]) * g1)

for n in range(0,N1+1):
    for k in range(0,n):
        zeta[k,n] = ((mp.power(tF[n] - tF[k], 1-a) - mp.power(tF[n] - tF[k+1], 1-a)) /
                     ((tF[k+1] - tF[k]) * g1))

for n in range(0,N1+1):
    for k in range(0,n):
        zeta_s[k,n] = ((mp.power(tF[n] - tF[k], 1-a_s) - mp.power(tF[n] - tF[k+1], 1-a_s)) /
                     ((tF[k+1] - tF[k]) * g1_s))


A1[0,0] = mp.mpf(1)/120   # first boundary condition
A1[0,1] = mp.mpf(26)/120
A1[0,2] = mp.mpf(66)/120
A1[0,3] = mp.mpf(26)/120
A1[0,4] = mp.mpf(1)/120

for i in range(M-2,M+3):
    A1[M+4,i+2]=A1[0,i-(M-2)]

A1[1,0]=-mp.mpf(717)/d1      # Equation with node x_0  # y-y"(0)
A1[1,1]=-mp.mpf(1448)/d1
A1[1,2]=mp.mpf(4300)/d1
A1[1,3]=-mp.mpf(1322)/d1
A1[1,4]=-mp.mpf(938)/d1
A1[1,5]=mp.mpf(202)/d1
A1[1,6]=-mp.mpf(92)/d1
A1[1,7]=mp.mpf(10)/d1
A1[1,8]=mp.mpf(7)/d1
A1[1,9]=-mp.mpf(2)/d1

for i in range(-7+M,M+3):
    A1[M+3,i+2]=A1[1,9-(i-(-7+M))]

A1[2,0]=-mp.mpf(1475)/c1         # Tau1
A1[2,1]=-mp.mpf(30149)/c1
A1[2,2]=mp.mpf(31918)/c1
A1[2,3]=mp.mpf(30322)/c1
A1[2,4]=-mp.mpf(27776)/c1
A1[2,5]=-mp.mpf(3680)/c1
A1[2,6]=mp.mpf(994)/c1
A1[2,7]=-mp.mpf(98)/c1
A1[2,8]=-mp.mpf(77)/c1
A1[2,9]=mp.mpf(21)/c1

for i in range(-7+M,M+3):
    A1[M+2,i+2]=A1[2,9-(i-(-7+M))]

A1[3,0]=mp.mpf(2)/d1                                # y-y"(1)
A1[3,1]=-mp.mpf(725)/d1
A1[3,2]=-mp.mpf(1454)/d1
A1[3,3]=mp.mpf(4396)/d1
A1[3,4]=-mp.mpf(1574)/d1
A1[3,5]=-mp.mpf(602)/d1
A1[3,6]=-mp.mpf(50)/d1
A1[3,7]=mp.mpf(4)/d1
A1[3,8]=mp.mpf(4)/d1
A1[3,9]=-mp.mpf(1)/d1

for i in range(-7+M,M+3):
    A1[M+1,i+2]=A1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    A1[i+2,-4+i+2]=mp.mpf(1)/d1
    A1[i+2,-3+i+2]=-mp.mpf(2)/d1
    A1[i+2,-2+i+2]=-mp.mpf(728)/d1
    A1[i+2,-1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,i+2]=mp.mpf(4270)/d1
    A1[i+2,1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,2+i+2]=-mp.mpf(728)/d1
    A1[i+2,3+i+2]=-mp.mpf(2)/d1
    A1[i+2,4+i+2]=mp.mpf(1)/d1

#------------------------------------------------------------------------------
# time level of first interval
B1[0,0]=A1[0,0]         # first boundary condition
B1[0,1]=A1[0,1]
B1[0,2]=A1[0,2]
B1[0,3]=A1[0,3]
B1[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B1[M+4,i+2]=B1[0,i-(M-2)]

# B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B1[1,1]=A1[1,1]+26*q1
# B1[1,2]=A1[1,2]+66*q1
# B1[1,3]=A1[1,3]+26*q1
# B1[1,4]=A1[1,4]+q1
B1[1,5]=A1[1,5]
B1[1,6]=A1[1,6]
B1[1,7]=A1[1,7]
B1[1,8]=A1[1,8]
B1[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

# B1[2,0]=A1[2,0]+q2         # Tau1
# B1[2,1]=A1[2,1]+237*q2
# B1[2,2]=A1[2,2]+1682*q2
# B1[2,3]=A1[2,3]+1682*q2
# B1[2,4]=A1[2,4]+237*q2
# B1[2,5]=A1[2,5]+q2
B1[2,6]=A1[2,6]
B1[2,7]=A1[2,7]
B1[2,8]=A1[2,8]
B1[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

B1[3,0]=A1[3,0]                                 # y-y"(1)
# B1[3,1]=A1[3,1]+q1
# B1[3,2]=A1[3,2]+26*q1
# B1[3,3]=A1[3,3]+66*q1
# B1[3,4]=A1[3,4]+26*q1
# B1[3,5]=A1[3,5]+q1
B1[3,6]=A1[3,6]
B1[3,7]=A1[3,7]
B1[3,8]=A1[3,8]
B1[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B1[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B1[i+2,3+i+2]=A1[i+2,3+i+2]
    B1[i+2,4+i+2]=A1[i+2,4+i+2]

for n in range(1,N1+1):
    q1=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(120)
    q2=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(3840)

    # B1[0,0]=A1[0,0]         # first boundary condition
    # B1[0,1]=A1[0,1]
    # B1[0,2]=A1[0,2]
    # B1[0,3]=A1[0,3]
    # B1[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B1[M+4,i+2]=B1[0,i-(M-2)]

    B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B1[1,1]=A1[1,1]+26*q1
    B1[1,2]=A1[1,2]+66*q1
    B1[1,3]=A1[1,3]+26*q1
    B1[1,4]=A1[1,4]+q1
    # B1[1,5]=A1[1,5]
    # B1[1,6]=A1[1,6]
    # B1[1,7]=A1[1,7]
    # B1[1,8]=A1[1,8]
    # B1[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

    B1[2,0]=A1[2,0]+q2         # Tau1
    B1[2,1]=A1[2,1]+237*q2
    B1[2,2]=A1[2,2]+1682*q2
    B1[2,3]=A1[2,3]+1682*q2
    B1[2,4]=A1[2,4]+237*q2
    B1[2,5]=A1[2,5]+q2
    # B1[2,6]=A1[2,6]
    # B1[2,7]=A1[2,7]
    # B1[2,8]=A1[2,8]
    # B1[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

    #B1[3,0]=A1[3,0]                                 # y-y"(1)
    B1[3,1]=A1[3,1]+q1
    B1[3,2]=A1[3,2]+26*q1
    B1[3,3]=A1[3,3]+66*q1
    B1[3,4]=A1[3,4]+26*q1
    B1[3,5]=A1[3,5]+q1
    # B1[3,6]=A1[3,6]
    # B1[3,7]=A1[3,7]
    # B1[3,8]=A1[3,8]
    # B1[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

    for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
        # B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B1[i+2,i+2]=A1[i+2,i+2]+66*q1
        B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B1[i+2,3+i+2]=A1[i+2,3+i+2]
        # B1[i+2,4+i+2]=A1[i+2,4+i+2]

    W1[0]=(tF[n]**2)*(mp.sin(2*mp.pi*x[0]))
    for i in range(0,M+3):
        s=0
        if n>1:
            s=mp.fdot(u[i,1:n]-u[i,0:n-1],zeta[0:n-1,n]+zeta_s[0:n-1,n])
        W1[i+1]=f(X[i],tF[n])+u[i,n-1]*(zeta[n-1,n]+zeta_s[n-1,n])-s
    W1[M+4]=(tF[n]**2)*(mp.sin(2*mp.pi*x[M]))


    # ab = extract_banded_form(B1, lower_bw=8, upper_bw=8)
    # G1 = solve_banded((8, 8), ab, W1)

    # for i in range(0,M+5):
    #     C1[n,i]=G1.item(i)

    B1_mp = mp.matrix(B1)
    W1_mp = mp.matrix(W1)

    G1 = mp.lu_solve(B1_mp, W1_mp)

    for i in range(M+5):
        C1[n, i] = G1[i]

    u[0,n]=(C1[n,0]+C1[n,1]*26+C1[n,2]*66+C1[n,3]*26+C1[n,4])/120
    u[1,n]=(C1[n,0]+C1[n,1]*237+C1[n,2]*1682+C1[n,3]*1682+C1[n,4]*237+C1[n,5])/3840

    for i in range(2,M+1):
        u[i,n]=(C1[n,i-3+2]+C1[n,i-2+2]*26+C1[n,i-1+2]*66+C1[n,i+2]*26+C1[n,i+1+2])/120

    u[M+1,n]=(C1[n,M-3+2]+237*C1[n,M-2+2]+1682*C1[n,M-1+2]+1682*C1[n,M+2]+237*C1[n,M+1+2]+C1[n,M+2+2])/3840
    u[M+2,n]=(C1[n,M-2+2]+C1[n,M-1+2]*26+C1[n,M+2]*66+C1[n,M+1+2]*26+C1[n,M+2+2])/120

#------------------------------------------------------------------------------------------------------
for i in range(0,M+3):
    U[i,1]=u[i,N1]

#Xi=np.zeros((N1+1,N+1))

# Xi = mp.matrix(N1+1, N+1)
# Xi_s = mp.matrix(N1+1, N+1)

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi[k,j]=(((t[j]-tF[k])**(1-a))-((t[j]-tF[k+1])**(1-a)))/((tF[k+1]-tF[k])*g1)

# #Xi_s=np.zeros((N1+1,N+1))

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi_s[k,j]=(((t[j]-tF[k])**(1-a_s))-((t[j]-tF[k+1])**(1-a_s)))/((tF[k+1]-tF[k])*g1_s)

Xi   = mp.matrix(N1 + 1, N + 1)
Xi_s = mp.matrix(N1 + 1, N + 1)

for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi[k, j] = (
            dt1**(1 - a) - dt2**(1 - a)
        ) / (hF * g1)


for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi_s[k, j] = (
            dt1**(1 - a_s) - dt2**(1 - a_s)
        ) / (hF * g1_s)


def gamma_mat(k,j):
    return ((mp.power(t[j] - t[k], 1 - a) - mp.power(t[j] - t[k+1], 1 - a)) / g1)

def gamma_mat_s(k,j):
    return ((mp.power(t[j] - t[k], 1 - a_s) - mp.power(t[j] - t[k+1], 1 - a_s)) / g1_s)

#Del=np.zeros((N+1,N+1))

#for j in range(0,N+1):
    #for k in range(0,j):
        #Del[k,j]=((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

#def Del(k,j):
    #return ((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

def Del(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a))
    term2 = mp.power(t[j] - t[k+1], 1 - a) * (t[j] + (a - 2) * t[k] + (1 - a) * t[k+1])
    return (term1 - term2) / g - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat(k, j)

def Del_s(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a_s))
    term2 = mp.power(t[j] - t[k+1], 1 - a_s) * (t[j] + (a_s - 2) * t[k] + (1 - a_s) * t[k+1])
    return (term1 - term2) / g_s - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat_s(k, j)

def d(k,j):
    if k == j:
        return (
            2 * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

def d_s(k,j):
    if k == j:
        return (
            2 * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat_s(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

B[0,0]=A1[0,0]         # first Boundary condition
B[0,1]=A1[0,1]
B[0,2]=A1[0,2]
B[0,3]=A1[0,3]
B[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B[M+4,i+2]=B[0,i-(M-2)]

# B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B[1,1]=A1[1,1]+26*q1
# B[1,2]=A1[1,2]+66*q1
# B[1,3]=A1[1,3]+26*q1
# B[1,4]=A1[1,4]+q1
B[1,5]=A1[1,5]
B[1,6]=A1[1,6]
B[1,7]=A1[1,7]
B[1,8]=A1[1,8]
B[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B[M+3,i+2]=B[1,9-(i-(-7+M))]

# B[2,0]=A1[2,0]+q2         # Tau1
# B[2,1]=A1[2,1]+237*q2
# B[2,2]=A1[2,2]+1682*q2
# B[2,3]=A1[2,3]+1682*q2
# B[2,4]=A1[2,4]+237*q2
# B[2,5]=A1[2,5]+q2
B[2,6]=A1[2,6]
B[2,7]=A1[2,7]
B[2,8]=A1[2,8]
B[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B[M+2,i+2]=B[2,9-(i-(-7+M))]

B[3,0]=A1[3,0]                                 # y-y"(1)
# B[3,1]=A1[3,1]+q1
# B[3,2]=A1[3,2]+26*q1
# B[3,3]=A1[3,3]+66*q1
# B[3,4]=A1[3,4]+26*q1
# B[3,5]=A1[3,5]+q1
B[3,6]=A1[3,6]
B[3,7]=A1[3,7]
B[3,8]=A1[3,8]
B[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B[i+2,3+i+2]=A1[i+2,3+i+2]
    B[i+2,4+i+2]=A1[i+2,4+i+2]


n=2
# --- D1 matrix ---
D1 = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1[0,0] = -two * mp.mpf(Del(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
D1[1,0] = two * Del(1, j) / (Tau[1] * Tau[2])
D1[2,0] = D1[0,0]+D1[1,0]

# --- D1 matrix ---
D1_s = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1_s[0,0] = -two * mp.mpf(Del_s(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
D1_s[1,0] = two * Del_s(1, j) / (Tau[1] * Tau[2])
D1_s[2,0] = D1_s[0,0]+D1_s[1,0]

q1=((D1[2,0]+D1_s[2,0])-c_)/120
q2=((D1[2,0]+D1_s[2,0])-c_)/3840

# B[0,0]=A1[0,0]         # first Boundary condition
# B[0,1]=A1[0,1]
# B[0,2]=A1[0,2]
# B[0,3]=A1[0,3]
# B[0,4]=A1[0,4]

# for i in range(M-2,M+3):
#     B[M+4,i+2]=B[0,i-(M-2)]

B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
B[1,1]=A1[1,1]+26*q1
B[1,2]=A1[1,2]+66*q1
B[1,3]=A1[1,3]+26*q1
B[1,4]=A1[1,4]+q1
# B[1,5]=A1[1,5]
# B[1,6]=A1[1,6]
# B[1,7]=A1[1,7]
# B[1,8]=A1[1,8]
# B[1,9]=A1[1,9]

for i in range(-7+M,M+3):
    B[M+3,i+2]=B[1,9-(i-(-7+M))]

B[2,0]=A1[2,0]+q2         # Tau1
B[2,1]=A1[2,1]+237*q2
B[2,2]=A1[2,2]+1682*q2
B[2,3]=A1[2,3]+1682*q2
B[2,4]=A1[2,4]+237*q2
B[2,5]=A1[2,5]+q2
# B[2,6]=A1[2,6]
# B[2,7]=A1[2,7]
# B[2,8]=A1[2,8]
# B[2,9]=A1[2,9]

for i in range(-7+M,M+3):
    B[M+2,i+2]=B[2,9-(i-(-7+M))]

# B[3,0]=A1[3,0]                                 # y-y"(1)
B[3,1]=A1[3,1]+q1
B[3,2]=A1[3,2]+26*q1
B[3,3]=A1[3,3]+66*q1
B[3,4]=A1[3,4]+26*q1
B[3,5]=A1[3,5]+q1
# B[3,6]=A1[3,6]
# B[3,7]=A1[3,7]
# B[3,8]=A1[3,8]
# B[3,9]=A1[3,9]

for i in range(-7+M,M+3):
    B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
    # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    B[i+2,i+2]=A1[i+2,i+2]+66*q1
    B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    # B[i+2,3+i+2]=A1[i+2,3+i+2]
    # B[i+2,4+i+2]=A1[i+2,4+i+2]

W[0]=(t[n]**2)*(mp.sin(2*mp.pi*x[0]))
for i in range(0,M+3):
    W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot(D1[0:n,0]+D1_s[0:n,0],U[i,0:n])+f(X[i],t[n])
W[M+4]=(t[n]**2)*(mp.sin(2*mp.pi*x[M]))

# ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
# G = solve_banded((8, 8), ab, W)

# for i in range(0,M+5):
#     C2[n,i]=G.item(i)

B_mp = mp.matrix(B)
W_mp = mp.matrix(W)

G = mp.lu_solve(B_mp, W_mp)

for i in range(M+5):
     C2[n, i] = G[i]

U[0,n]=(C2[n,0]+C2[n,1]*26+C2[n,2]*66+C2[n,3]*26+C2[n,4])/120
U[1,n]=(C2[n,0]+C2[n,1]*237+C2[n,2]*1682+C2[n,3]*1682+C2[n,4]*237+C2[n,5])/3840

for i in range(2,M+1):
    U[i,n]=(C2[n,i-3+2]+C2[n,i-2+2]*26+C2[n,i-1+2]*66+C2[n,i+2]*26+C2[n,i+1+2])/120

U[M+1,n]=(C2[n,M-3+2]+237*C2[n,M-2+2]+1682*C2[n,M-1+2]+1682*C2[n,M+2]+237*C2[n,M+1+2]+C2[n,M+2+2])/3840
U[M+2,n]=(C2[n,M-2+2]+C2[n,M-1+2]*26+C2[n,M+2]*66+C2[n,M+1+2]*26+C2[n,M+2+2])/120


for n in range(3,N+1):
    # --- D1 matrix ---
    D1 = mp.zeros(N+1, 1)  # ✅ allows mpf/mpc objects
    two=mp.mpf(2)
    j=n
    D1[0,0] = -two * Del(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1[1,0] = (
        -two * Del(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat(2, j) / (Tau[2] + Tau[3])
    )

    if j>3:
        for k in range(2, j - 1):
            D1[k,0] = (
                -two * Del(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
                + two * Del(k, j) / (Tau[k] * Tau[k + 1])
                - two * Del(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
                - gamma_mat(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
                + gamma_mat(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
            )

    D1[j - 1,0] = (
        -two * Del(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1[j,0] = (
        two * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    # --- D1 matrix ---
    D1_s = mp.zeros(N+1,1)
    two=mp.mpf(2)
    j=n
    D1_s[0,0] = -two * Del_s(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1_s[1,0] = (
        -two * Del_s(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del_s(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat_s(2, j) / (Tau[2] + Tau[3])
    )
    for k in range(2, j - 1):
        D1_s[k,0] = (
            -two * Del_s(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
            + two * Del_s(k, j) / (Tau[k] * Tau[k + 1])
            - two * Del_s(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
            - gamma_mat_s(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
            + gamma_mat_s(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
        )
    D1_s[j - 1,0] = (
        -two * Del_s(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del_s(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat_s(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1_s[j,0] = (
        two * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat_s(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    q1=((d(n,n)+d_s(n,n))-c_)/120
    q2=((d(n,n)+d_s(n,n))-c_)/3840

    # B[0,0]=A1[0,0]         # first Boundary condition
    # B[0,1]=A1[0,1]
    # B[0,2]=A1[0,2]
    # B[0,3]=A1[0,3]
    # B[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B[M+4,i+2]=B[0,i-(M-2)]

    B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1
    # B[1,5]=A1[1,5]
    # B[1,6]=A1[1,6]
    # B[1,7]=A1[1,7]
    # B[1,8]=A1[1,8]
    # B[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2
    # B[2,6]=A1[2,6]
    # B[2,7]=A1[2,7]
    # B[2,8]=A1[2,8]
    # B[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    # B[3,0]=A1[3,0]                                 # y-y"(1)
    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1
    # B[3,6]=A1[3,6]
    # B[3,7]=A1[3,7]
    # B[3,8]=A1[3,8]
    # B[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
        # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B[i+2,3+i+2]=A1[i+2,3+i+2]
        # B[i+2,4+i+2]=A1[i+2,4+i+2]

    W[0]=(t[n]**2)*(mp.sin(2*mp.pi*x[0]))
    for i in range(0,M+3):
        W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot((D1[0:n,0]+D1_s[0:n,0]),U[i,0:n])+f(X[i],t[n])
    W[M+4]=(t[n]**2)*(mp.sin(2*mp.pi*x[M]))

    # ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
    # G = solve_banded((8, 8), ab, W)

    # for i in range(0,M+5):
    #     C[n,i]=G.item(i)

    B_mp = mp.matrix(B)
    W_mp = mp.matrix(W)

    G = mp.lu_solve(B_mp, W_mp)

    for i in range(M+5):
         C[n, i] = G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

    U[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
    U[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

# u_exact=np.zeros((M+3,N1+1))
# u_error=np.zeros((M+3,N1+1))
# for j in range(0,N1+1):
#     for i in range(0,M+3):
#         u_exact[i,j]=(tF[j]**2)*(mp.sin(2*mp.pi*X[i]))
#         u_error[i,j]=abs(u_exact[i,j]-u[i,j])

# U_exact=np.zeros((M+3,N+1))
# U_error=np.zeros((M+3,N+1))
U_exact = mp.matrix(M+3, N+1)
U_error = mp.matrix(M+3, N+1)

for i in range(0,M+3):
    for j in range(0,N+1):
        U_exact[i,j]=(t[j]**2)*(mp.sin(2*mp.pi*X[i]))
        U_error[i,j]=abs(U_exact[i,j]-U[i,j])

#print(np.max(U_error))
max_error = max(abs(U_error[i,j]) for i in range(U_error.rows) for j in range(U_error.cols))
print(max_error)

# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Meshgrid (coarse grid)
# # --------------------------------------------------
# Xplot_coarse, Yplot_coarse = np.meshgrid(X, t)

# Xplot_coarse = Xplot_coarse.astype(float)
# Yplot_coarse = Yplot_coarse.astype(float)
# V_float1 = np.array(U_exact, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float1.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Exact solution")
# ax.grid(False)

# # Fixed good viewing angle (for paper)
# ax.view_init(elev=28, azim=225)


# # --------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("exact_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("exact_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -------------------------------------------------------------------------------------------------------------------------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Convert numerical solution to float
# # --------------------------------------------------
# V_float = np.array(U, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Numerical solution ")
# ax.grid(False)

# # Fixed good viewing angle (match exact solution plot)
# ax.view_init(elev=28, azim=225)

# # -------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("numerical_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("numerical_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "lines.linewidth": 2,
#     "figure.dpi": 120
# })

# #-------------------------------------------------------------------------------------------------------------------------------------

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# U_error_float = np.array(U_error, dtype=float)

# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, U_error_float.T,
#     cmap='viridis',
#     linewidth=0,
#     antialiased=True,
#     shade=True
# )

# surf.set_rasterized(True)

# ax.set_xlabel(r"Space $x$", labelpad=10)
# ax.set_ylabel(r"Time $t$", labelpad=10)
# ax.set_zlabel(r"Absolute error", labelpad=10)

# ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# ax.ticklabel_format(axis='z', style='sci', scilimits=(0,0))

# ax.view_init(elev=28, azim=225)
# ax.dist = 11
# ax.grid(False)

# plt.tight_layout()
# plt.savefig("Error.png", dpi=300, bbox_inches="tight")
# plt.savefig("Error.pdf", bbox_inches="tight")
# plt.show()

# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)

# end=datetime.now()
# print(end-start)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from datetime import datetime
import mpmath as mp

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 120,
})

# --------------------------------------------------
# mpmath -> numpy float conversion
# --------------------------------------------------
def to_float(a, vector=False):
    """Convert mp.matrix / list of mpf / nested lists to a float ndarray."""
    if isinstance(a, mp.matrix):
        a = a.tolist()
    arr = np.array([[float(v) for v in row] for row in a]) \
        if (len(a) and isinstance(a[0], (list, tuple))) \
        else np.array([float(v) for v in a])
    return arr.ravel() if vector else arr

X_f = to_float(X, vector=True)
t_f = to_float(t, vector=True)

Xplot_coarse, Yplot_coarse = np.meshgrid(X_f, t_f)   # shape (len(t), len(X))

U_exact_f = to_float(U_exact)   # expected shape (len(X), len(t))
U_f       = to_float(U)
U_err_f   = to_float(U_error)

# --------------------------------------------------
# Generic surface plot
# --------------------------------------------------
def surface(Z, zlabel, fname, sci=False, animate=False):
    fig = plt.figure(figsize=(7.5, 5.5))
    ax = fig.add_subplot(111, projection="3d")

    surf = ax.plot_surface(Xplot_coarse, Yplot_coarse, Z.T,
                           cmap="viridis", linewidth=0,
                           antialiased=True, shade=True)
    surf.set_rasterized(True)

    ax.set_xlabel(r"Space $x$", labelpad=10)
    ax.set_ylabel(r"Time $t$", labelpad=10)
    ax.set_zlabel(zlabel, labelpad=10)
    if sci:
        ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
        ax.ticklabel_format(axis="z", style="sci", scilimits=(0, 0))

    ax.view_init(elev=28, azim=225)
    ax.set_box_aspect(None, zoom=0.9)   # replaces deprecated ax.dist
    ax.grid(False)

    plt.tight_layout()
    plt.savefig(f"{fname}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{fname}.pdf", bbox_inches="tight")
    plt.show()

    if animate:
        for angle in range(0, 360, 5):
            ax.view_init(elev=30, azim=angle)
            plt.draw()
            plt.pause(0.1)

surface(U_exact_f, r"Exact solution",     "exact_solution_surface_alpha06")
surface(U_f,       r"Numerical solution", "numerical_solution_surface_alpha06")
surface(U_err_f,   r"Absolute error",     "Error", sci=True)

end = datetime.now()
print(end - start)













Example_1
code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
#from scipy.linalg import solve
from datetime import datetime
start = datetime.now()
from scipy.linalg import solve_banded
from mpmath import mp
#from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import ScalarFormatter

mp.dps = 120;

a = mp.mpf("0.80")
a_s = mp.mpf("0.40")

g1 = mp.gamma(2 - a)
g1_s = mp.gamma(2 - a_s)
g = mp.gamma(3 - a)
g_s = mp.gamma(3 - a_s)

def f(x, t):
    x = mp.mpf(x)
    t = mp.mpf(t)
    return (mp.power(t,2-a)*(1/g)+mp.power(t,2-a_s)*(1/g_s))*2*(x-x**2)+2*(1+t**2)

a_ = mp.mpf('1')
c_ = mp.mpf('0')
L  = mp.mpf('1')
T  = mp.mpf('1')

M=8
#N=math.floor(T*((M/L)**(6/(3-a))))
N=5000
h = mp.mpf(L) / mp.mpf(M)
#r=mp.mpf(3/a)-1
r=mp.mpf(1)

#t=[T*((i/N)**r) for i in range(0,N+1)]
#t = T * (np.linspace(0, 1, N+1) ** r)
t = [T * (mp.mpf(i) / N) ** r for i in range(N + 1)]

#N1=math.ceil((mp.power(N,a-3)/t[1]**(2-a))**(1/(a-2)))
N1 = int(mp.ceil(
    mp.power(
        mp.power(N, a - 3) / mp.power(t[1], 2 - a),
        1 / (a - 2)
    )
))

#tF=[t[1]*((i/N1)**r) for i in range(0,N1+1)]
#tF = t[1] * np.linspace(0, 1, N1+1) ** r
tF = [t[1] * mp.power(mp.mpf(i) / N1, r) for i in range(N1 + 1)]

# Tau=np.zeros(N+1)
# for i in range(1,N+1):
#     Tau[i]=t[i]-t[i-1]
#Tau = np.diff(t, prepend=0)  # prepend=0  t[−1] = 0
Tau = [mp.mpf('0')] + [t[i] - t[i-1] for i in range(1, len(t))]

#x=[i*h for i in range(0,M+1)]
# x = np.linspace(0, L, M+1)
# X=np.zeros(M+3)

x = [mp.mpf(i) * L / M for i in range(M+1)]
X = [mp.mpf('0') for _ in range(M+3)]

X[0]=x[0]
X[1]=(x[0]+x[1])/2
for j in range(2,M+1):
    X[j]=x[j-1]
X[M+1]=(x[M]+x[M-1])/2
X[M+2]=x[M]

# u=np.zeros((M+3,N1+1))
# U=np.zeros((M+3,N+1))
# exact=np.zeros((M+3,N+1))
# error=np.zeros((M+3,N+1))

# A1 = np.zeros((M+5, M+5))
# B1=np.zeros((M+5,M+5))
# B=np.zeros((M+5,M+5))
# C=np.zeros((N+1,M+5))
# C1=np.zeros((N+1,M+5))
# C2=np.zeros((N+1,M+5))
# W1=np.zeros(M+5)
# W=np.zeros(M+5)

u     = mp.matrix(M+3, N1+1)
U     = mp.matrix(M+3, N+1)
exact = mp.matrix(M+3, N+1)
error = mp.matrix(M+3, N+1)

A1 = mp.matrix(M+5, M+5)
B1 = mp.matrix(M+5, M+5)
B  = mp.matrix(M+5, M+5)

C  = mp.matrix(N+1, M+5)
C1 = mp.matrix(N+1, M+5)
C2 = mp.matrix(N+1, M+5)

W1 = mp.matrix(M+5, 1)
W  = mp.matrix(M+5, 1)

zeta   = mp.matrix(N1+1, N1+1)
zeta_s = mp.matrix(N1+1, N1+1)

c1 = mp.mpf(69120) * h**2 / mp.mpf(a_)
d1 = mp.mpf(4320) * h**2 / mp.mpf(a_)

# --- Compute zeta ---
# for n in range(1, N1 + 1):
#     dt = tF[n] - tF[:n]
#     dt1 = tF[n] - tF[1:n + 1]
#     zeta[:n, n] = (dt**(1 - a) - dt1**(1 - a)) / ((tF[1:n + 1] - tF[:n]) * g1)

for n in range(0,N1+1):
    for k in range(0,n):
        zeta[k,n] = ((mp.power(tF[n] - tF[k], 1-a) - mp.power(tF[n] - tF[k+1], 1-a)) /
                     ((tF[k+1] - tF[k]) * g1))

for n in range(0,N1+1):
    for k in range(0,n):
        zeta_s[k,n] = ((mp.power(tF[n] - tF[k], 1-a_s) - mp.power(tF[n] - tF[k+1], 1-a_s)) /
                     ((tF[k+1] - tF[k]) * g1_s))

for i in range(0,M+3):
    u[i,0]= X[i]-X[i]**2
    U[i,0]=u[i,0]

A1[0,0] = mp.mpf(1)/120   # first boundary condition
A1[0,1] = mp.mpf(26)/120
A1[0,2] = mp.mpf(66)/120
A1[0,3] = mp.mpf(26)/120
A1[0,4] = mp.mpf(1)/120

for i in range(M-2,M+3):
    A1[M+4,i+2]=A1[0,i-(M-2)]

A1[1,0]=-mp.mpf(717)/d1      # Equation with node x_0  # y-y"(0)
A1[1,1]=-mp.mpf(1448)/d1
A1[1,2]=mp.mpf(4300)/d1
A1[1,3]=-mp.mpf(1322)/d1
A1[1,4]=-mp.mpf(938)/d1
A1[1,5]=mp.mpf(202)/d1
A1[1,6]=-mp.mpf(92)/d1
A1[1,7]=mp.mpf(10)/d1
A1[1,8]=mp.mpf(7)/d1
A1[1,9]=-mp.mpf(2)/d1

for i in range(-7+M,M+3):
    A1[M+3,i+2]=A1[1,9-(i-(-7+M))]

A1[2,0]=-mp.mpf(1475)/c1         # Tau1
A1[2,1]=-mp.mpf(30149)/c1
A1[2,2]=mp.mpf(31918)/c1
A1[2,3]=mp.mpf(30322)/c1
A1[2,4]=-mp.mpf(27776)/c1
A1[2,5]=-mp.mpf(3680)/c1
A1[2,6]=mp.mpf(994)/c1
A1[2,7]=-mp.mpf(98)/c1
A1[2,8]=-mp.mpf(77)/c1
A1[2,9]=mp.mpf(21)/c1

for i in range(-7+M,M+3):
    A1[M+2,i+2]=A1[2,9-(i-(-7+M))]

A1[3,0]=mp.mpf(2)/d1                                # y-y"(1)
A1[3,1]=-mp.mpf(725)/d1
A1[3,2]=-mp.mpf(1454)/d1
A1[3,3]=mp.mpf(4396)/d1
A1[3,4]=-mp.mpf(1574)/d1
A1[3,5]=-mp.mpf(602)/d1
A1[3,6]=-mp.mpf(50)/d1
A1[3,7]=mp.mpf(4)/d1
A1[3,8]=mp.mpf(4)/d1
A1[3,9]=-mp.mpf(1)/d1

for i in range(-7+M,M+3):
    A1[M+1,i+2]=A1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    A1[i+2,-4+i+2]=mp.mpf(1)/d1
    A1[i+2,-3+i+2]=-mp.mpf(2)/d1
    A1[i+2,-2+i+2]=-mp.mpf(728)/d1
    A1[i+2,-1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,i+2]=mp.mpf(4270)/d1
    A1[i+2,1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,2+i+2]=-mp.mpf(728)/d1
    A1[i+2,3+i+2]=-mp.mpf(2)/d1
    A1[i+2,4+i+2]=mp.mpf(1)/d1

#------------------------------------------------------------------------------
# time level of first interval
B1[0,0]=A1[0,0]         # first boundary condition
B1[0,1]=A1[0,1]
B1[0,2]=A1[0,2]
B1[0,3]=A1[0,3]
B1[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B1[M+4,i+2]=B1[0,i-(M-2)]

# B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B1[1,1]=A1[1,1]+26*q1
# B1[1,2]=A1[1,2]+66*q1
# B1[1,3]=A1[1,3]+26*q1
# B1[1,4]=A1[1,4]+q1
B1[1,5]=A1[1,5]
B1[1,6]=A1[1,6]
B1[1,7]=A1[1,7]
B1[1,8]=A1[1,8]
B1[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

# B1[2,0]=A1[2,0]+q2         # Tau1
# B1[2,1]=A1[2,1]+237*q2
# B1[2,2]=A1[2,2]+1682*q2
# B1[2,3]=A1[2,3]+1682*q2
# B1[2,4]=A1[2,4]+237*q2
# B1[2,5]=A1[2,5]+q2
B1[2,6]=A1[2,6]
B1[2,7]=A1[2,7]
B1[2,8]=A1[2,8]
B1[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

B1[3,0]=A1[3,0]                                 # y-y"(1)
# B1[3,1]=A1[3,1]+q1
# B1[3,2]=A1[3,2]+26*q1
# B1[3,3]=A1[3,3]+66*q1
# B1[3,4]=A1[3,4]+26*q1
# B1[3,5]=A1[3,5]+q1
B1[3,6]=A1[3,6]
B1[3,7]=A1[3,7]
B1[3,8]=A1[3,8]
B1[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B1[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B1[i+2,3+i+2]=A1[i+2,3+i+2]
    B1[i+2,4+i+2]=A1[i+2,4+i+2]

for n in range(1,N1+1):
    q1=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(120)
    q2=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(3840)

    # B1[0,0]=A1[0,0]         # first boundary condition
    # B1[0,1]=A1[0,1]
    # B1[0,2]=A1[0,2]
    # B1[0,3]=A1[0,3]
    # B1[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B1[M+4,i+2]=B1[0,i-(M-2)]

    B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B1[1,1]=A1[1,1]+26*q1
    B1[1,2]=A1[1,2]+66*q1
    B1[1,3]=A1[1,3]+26*q1
    B1[1,4]=A1[1,4]+q1
    # B1[1,5]=A1[1,5]
    # B1[1,6]=A1[1,6]
    # B1[1,7]=A1[1,7]
    # B1[1,8]=A1[1,8]
    # B1[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

    B1[2,0]=A1[2,0]+q2         # Tau1
    B1[2,1]=A1[2,1]+237*q2
    B1[2,2]=A1[2,2]+1682*q2
    B1[2,3]=A1[2,3]+1682*q2
    B1[2,4]=A1[2,4]+237*q2
    B1[2,5]=A1[2,5]+q2
    # B1[2,6]=A1[2,6]
    # B1[2,7]=A1[2,7]
    # B1[2,8]=A1[2,8]
    # B1[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

    #B1[3,0]=A1[3,0]                                 # y-y"(1)
    B1[3,1]=A1[3,1]+q1
    B1[3,2]=A1[3,2]+26*q1
    B1[3,3]=A1[3,3]+66*q1
    B1[3,4]=A1[3,4]+26*q1
    B1[3,5]=A1[3,5]+q1
    # B1[3,6]=A1[3,6]
    # B1[3,7]=A1[3,7]
    # B1[3,8]=A1[3,8]
    # B1[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

    for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
        # B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B1[i+2,i+2]=A1[i+2,i+2]+66*q1
        B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B1[i+2,3+i+2]=A1[i+2,3+i+2]
        # B1[i+2,4+i+2]=A1[i+2,4+i+2]

    W1[0]=(1+tF[n]**2)*(x[0]-x[0]**2)
    for i in range(0,M+3):
        s=0
        if n>1:
            s=mp.fdot(u[i,1:n]-u[i,0:n-1],zeta[0:n-1,n]+zeta_s[0:n-1,n])
        W1[i+1]=f(X[i],tF[n])+u[i,n-1]*(zeta[n-1,n]+zeta_s[n-1,n])-s
    W1[M+4]=(1+tF[n]**2)*(x[M]-x[M]**2)


    # ab = extract_banded_form(B1, lower_bw=8, upper_bw=8)
    # G1 = solve_banded((8, 8), ab, W1)

    # for i in range(0,M+5):
    #     C1[n,i]=G1.item(i)

    B1_mp = mp.matrix(B1)
    W1_mp = mp.matrix(W1)

    G1 = mp.lu_solve(B1_mp, W1_mp)

    for i in range(M+5):
        C1[n, i] = G1[i]

    u[0,n]=(C1[n,0]+C1[n,1]*26+C1[n,2]*66+C1[n,3]*26+C1[n,4])/120
    u[1,n]=(C1[n,0]+C1[n,1]*237+C1[n,2]*1682+C1[n,3]*1682+C1[n,4]*237+C1[n,5])/3840

    for i in range(2,M+1):
        u[i,n]=(C1[n,i-3+2]+C1[n,i-2+2]*26+C1[n,i-1+2]*66+C1[n,i+2]*26+C1[n,i+1+2])/120

    u[M+1,n]=(C1[n,M-3+2]+237*C1[n,M-2+2]+1682*C1[n,M-1+2]+1682*C1[n,M+2]+237*C1[n,M+1+2]+C1[n,M+2+2])/3840
    u[M+2,n]=(C1[n,M-2+2]+C1[n,M-1+2]*26+C1[n,M+2]*66+C1[n,M+1+2]*26+C1[n,M+2+2])/120

#------------------------------------------------------------------------------------------------------
for i in range(0,M+3):
    U[i,1]=u[i,N1]

#Xi=np.zeros((N1+1,N+1))

# Xi = mp.matrix(N1+1, N+1)
# Xi_s = mp.matrix(N1+1, N+1)

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi[k,j]=(((t[j]-tF[k])**(1-a))-((t[j]-tF[k+1])**(1-a)))/((tF[k+1]-tF[k])*g1)

# #Xi_s=np.zeros((N1+1,N+1))

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi_s[k,j]=(((t[j]-tF[k])**(1-a_s))-((t[j]-tF[k+1])**(1-a_s)))/((tF[k+1]-tF[k])*g1_s)

Xi   = mp.matrix(N1 + 1, N + 1)
Xi_s = mp.matrix(N1 + 1, N + 1)

for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi[k, j] = (
            dt1**(1 - a) - dt2**(1 - a)
        ) / (hF * g1)


for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi_s[k, j] = (
            dt1**(1 - a_s) - dt2**(1 - a_s)
        ) / (hF * g1_s)


def gamma_mat(k,j):
    return ((mp.power(t[j] - t[k], 1 - a) - mp.power(t[j] - t[k+1], 1 - a)) / g1)

def gamma_mat_s(k,j):
    return ((mp.power(t[j] - t[k], 1 - a_s) - mp.power(t[j] - t[k+1], 1 - a_s)) / g1_s)

#Del=np.zeros((N+1,N+1))

#for j in range(0,N+1):
    #for k in range(0,j):
        #Del[k,j]=((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

#def Del(k,j):
    #return ((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

def Del(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a))
    term2 = mp.power(t[j] - t[k+1], 1 - a) * (t[j] + (a - 2) * t[k] + (1 - a) * t[k+1])
    return (term1 - term2) / g - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat(k, j)

def Del_s(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a_s))
    term2 = mp.power(t[j] - t[k+1], 1 - a_s) * (t[j] + (a_s - 2) * t[k] + (1 - a_s) * t[k+1])
    return (term1 - term2) / g_s - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat_s(k, j)

def d(k,j):
    if k == j:
        return (
            2 * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

def d_s(k,j):
    if k == j:
        return (
            2 * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat_s(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

B[0,0]=A1[0,0]         # first Boundary condition
B[0,1]=A1[0,1]
B[0,2]=A1[0,2]
B[0,3]=A1[0,3]
B[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B[M+4,i+2]=B[0,i-(M-2)]

# B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B[1,1]=A1[1,1]+26*q1
# B[1,2]=A1[1,2]+66*q1
# B[1,3]=A1[1,3]+26*q1
# B[1,4]=A1[1,4]+q1
B[1,5]=A1[1,5]
B[1,6]=A1[1,6]
B[1,7]=A1[1,7]
B[1,8]=A1[1,8]
B[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B[M+3,i+2]=B[1,9-(i-(-7+M))]

# B[2,0]=A1[2,0]+q2         # Tau1
# B[2,1]=A1[2,1]+237*q2
# B[2,2]=A1[2,2]+1682*q2
# B[2,3]=A1[2,3]+1682*q2
# B[2,4]=A1[2,4]+237*q2
# B[2,5]=A1[2,5]+q2
B[2,6]=A1[2,6]
B[2,7]=A1[2,7]
B[2,8]=A1[2,8]
B[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B[M+2,i+2]=B[2,9-(i-(-7+M))]

B[3,0]=A1[3,0]                                 # y-y"(1)
# B[3,1]=A1[3,1]+q1
# B[3,2]=A1[3,2]+26*q1
# B[3,3]=A1[3,3]+66*q1
# B[3,4]=A1[3,4]+26*q1
# B[3,5]=A1[3,5]+q1
B[3,6]=A1[3,6]
B[3,7]=A1[3,7]
B[3,8]=A1[3,8]
B[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B[i+2,3+i+2]=A1[i+2,3+i+2]
    B[i+2,4+i+2]=A1[i+2,4+i+2]


n=2
# --- D1 matrix ---
D1 = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1[0,0] = -two * mp.mpf(Del(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
D1[1,0] = two * Del(1, j) / (Tau[1] * Tau[2])
D1[2,0] = D1[0,0]+D1[1,0]

# --- D1 matrix ---
D1_s = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1_s[0,0] = -two * mp.mpf(Del_s(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
D1_s[1,0] = two * Del_s(1, j) / (Tau[1] * Tau[2])
D1_s[2,0] = D1_s[0,0]+D1_s[1,0]

q1=((D1[2,0]+D1_s[2,0])-c_)/120
q2=((D1[2,0]+D1_s[2,0])-c_)/3840

# B[0,0]=A1[0,0]         # first Boundary condition
# B[0,1]=A1[0,1]
# B[0,2]=A1[0,2]
# B[0,3]=A1[0,3]
# B[0,4]=A1[0,4]

# for i in range(M-2,M+3):
#     B[M+4,i+2]=B[0,i-(M-2)]

B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
B[1,1]=A1[1,1]+26*q1
B[1,2]=A1[1,2]+66*q1
B[1,3]=A1[1,3]+26*q1
B[1,4]=A1[1,4]+q1
# B[1,5]=A1[1,5]
# B[1,6]=A1[1,6]
# B[1,7]=A1[1,7]
# B[1,8]=A1[1,8]
# B[1,9]=A1[1,9]

for i in range(-7+M,M+3):
    B[M+3,i+2]=B[1,9-(i-(-7+M))]

B[2,0]=A1[2,0]+q2         # Tau1
B[2,1]=A1[2,1]+237*q2
B[2,2]=A1[2,2]+1682*q2
B[2,3]=A1[2,3]+1682*q2
B[2,4]=A1[2,4]+237*q2
B[2,5]=A1[2,5]+q2
# B[2,6]=A1[2,6]
# B[2,7]=A1[2,7]
# B[2,8]=A1[2,8]
# B[2,9]=A1[2,9]

for i in range(-7+M,M+3):
    B[M+2,i+2]=B[2,9-(i-(-7+M))]

# B[3,0]=A1[3,0]                                 # y-y"(1)
B[3,1]=A1[3,1]+q1
B[3,2]=A1[3,2]+26*q1
B[3,3]=A1[3,3]+66*q1
B[3,4]=A1[3,4]+26*q1
B[3,5]=A1[3,5]+q1
# B[3,6]=A1[3,6]
# B[3,7]=A1[3,7]
# B[3,8]=A1[3,8]
# B[3,9]=A1[3,9]

for i in range(-7+M,M+3):
    B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
    # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    B[i+2,i+2]=A1[i+2,i+2]+66*q1
    B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    # B[i+2,3+i+2]=A1[i+2,3+i+2]
    # B[i+2,4+i+2]=A1[i+2,4+i+2]

W[0]=(1+t[n]**2)*(x[0]-x[0]**2)
for i in range(0,M+3):
    W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot(D1[0:n,0]+D1_s[0:n,0],U[i,0:n])+f(X[i],t[n])
W[M+4]=(1+t[n]**2)*(x[M]-x[M]**2)

# ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
# G = solve_banded((8, 8), ab, W)

# for i in range(0,M+5):
#     C2[n,i]=G.item(i)

B_mp = mp.matrix(B)
W_mp = mp.matrix(W)

G = mp.lu_solve(B_mp, W_mp)

for i in range(M+5):
     C2[n, i] = G[i]

U[0,n]=(C2[n,0]+C2[n,1]*26+C2[n,2]*66+C2[n,3]*26+C2[n,4])/120
U[1,n]=(C2[n,0]+C2[n,1]*237+C2[n,2]*1682+C2[n,3]*1682+C2[n,4]*237+C2[n,5])/3840

for i in range(2,M+1):
    U[i,n]=(C2[n,i-3+2]+C2[n,i-2+2]*26+C2[n,i-1+2]*66+C2[n,i+2]*26+C2[n,i+1+2])/120

U[M+1,n]=(C2[n,M-3+2]+237*C2[n,M-2+2]+1682*C2[n,M-1+2]+1682*C2[n,M+2]+237*C2[n,M+1+2]+C2[n,M+2+2])/3840
U[M+2,n]=(C2[n,M-2+2]+C2[n,M-1+2]*26+C2[n,M+2]*66+C2[n,M+1+2]*26+C2[n,M+2+2])/120


for n in range(3,N+1):
    # --- D1 matrix ---
    D1 = mp.zeros(N+1, 1)
    two=mp.mpf(2)
    j=n
    D1[0,0] = -two * Del(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1[1,0] = (
        -two * Del(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat(2, j) / (Tau[2] + Tau[3])
    )

    if j>3:
        for k in range(2, j - 1):
            D1[k,0] = (
                -two * Del(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
                + two * Del(k, j) / (Tau[k] * Tau[k + 1])
                - two * Del(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
                - gamma_mat(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
                + gamma_mat(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
            )

    D1[j - 1,0] = (
        -two * Del(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1[j,0] = (
        two * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    # --- D1 matrix ---
    D1_s = mp.zeros(N+1,1)
    two=mp.mpf(2)
    j=n
    D1_s[0,0] = -two * Del_s(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1_s[1,0] = (
        -two * Del_s(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del_s(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat_s(2, j) / (Tau[2] + Tau[3])
    )
    for k in range(2, j - 1):
        D1_s[k,0] = (
            -two * Del_s(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
            + two * Del_s(k, j) / (Tau[k] * Tau[k + 1])
            - two * Del_s(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
            - gamma_mat_s(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
            + gamma_mat_s(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
        )
    D1_s[j - 1,0] = (
        -two * Del_s(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del_s(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat_s(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1_s[j,0] = (
        two * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat_s(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    q1=((d(n,n)+d_s(n,n))-c_)/120
    q2=((d(n,n)+d_s(n,n))-c_)/3840

    # B[0,0]=A1[0,0]         # first Boundary condition
    # B[0,1]=A1[0,1]
    # B[0,2]=A1[0,2]
    # B[0,3]=A1[0,3]
    # B[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B[M+4,i+2]=B[0,i-(M-2)]

    B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1
    # B[1,5]=A1[1,5]
    # B[1,6]=A1[1,6]
    # B[1,7]=A1[1,7]
    # B[1,8]=A1[1,8]
    # B[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2
    # B[2,6]=A1[2,6]
    # B[2,7]=A1[2,7]
    # B[2,8]=A1[2,8]
    # B[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    # B[3,0]=A1[3,0]                                 # y-y"(1)
    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1
    # B[3,6]=A1[3,6]
    # B[3,7]=A1[3,7]
    # B[3,8]=A1[3,8]
    # B[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
        # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B[i+2,3+i+2]=A1[i+2,3+i+2]
        # B[i+2,4+i+2]=A1[i+2,4+i+2]

    W[0]=(1+t[n]**2)*(x[0]-x[0]**2)
    for i in range(0,M+3):
        W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot((D1[0:n,0]+D1_s[0:n,0]),U[i,0:n])+f(X[i],t[n])
    W[M+4]=(1+t[n]**2)*(x[M]-x[M]**2)

    # ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
    # G = solve_banded((8, 8), ab, W)

    # for i in range(0,M+5):
    #     C[n,i]=G.item(i)

    B_mp = mp.matrix(B)
    W_mp = mp.matrix(W)

    G = mp.lu_solve(B_mp, W_mp)

    for i in range(M+5):
         C[n, i] = G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

    U[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
    U[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

# u_exact=np.zeros((M+3,N1+1))
# u_error=np.zeros((M+3,N1+1))
# for j in range(0,N1+1):
#     for i in range(0,M+3):
#         u_exact[i,j]=(1+tF[j]**2)*(X[i]-X[i]**2)
#         u_error[i,j]=abs(u_exact[i,j]-u[i,j])

# U_exact=np.zeros((M+3,N+1))
# U_error=np.zeros((M+3,N+1))
U_exact = mp.matrix(M+3, N+1)
U_error = mp.matrix(M+3, N+1)

for i in range(0,M+3):
    for j in range(0,N+1):
        U_exact[i,j]=(1+t[j]**2)*(X[i]-X[i]**2)
        U_error[i,j]=abs(U_exact[i,j]-U[i,j])

#print(np.max(U_error))
max_error = max(abs(U_error[i,j]) for i in range(U_error.rows) for j in range(U_error.cols))
print(max_error)

# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Meshgrid (coarse grid)
# # --------------------------------------------------
# Xplot_coarse, Yplot_coarse = np.meshgrid(X, t)

# Xplot_coarse = Xplot_coarse.astype(float)
# Yplot_coarse = Yplot_coarse.astype(float)
# V_float1 = np.array(U_exact, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float1.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Exact solution")
# ax.grid(False)

# # Fixed good viewing angle (for paper)
# ax.view_init(elev=28, azim=225)


# # --------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("exact_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("exact_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -------------------------------------------------------------------------------------------------------------------------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Convert numerical solution to float
# # --------------------------------------------------
# V_float = np.array(U, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Numerical solution ")
# ax.grid(False)

# # Fixed good viewing angle (match exact solution plot)
# ax.view_init(elev=28, azim=225)

# # -------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("numerical_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("numerical_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "lines.linewidth": 2,
#     "figure.dpi": 120
# })

# #-------------------------------------------------------------------------------------------------------------------------------------

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# U_error_float = np.array(U_error, dtype=float)

# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, U_error_float.T,
#     cmap='viridis',
#     linewidth=0,
#     antialiased=True,
#     shade=True
# )

# surf.set_rasterized(True)

# ax.set_xlabel(r"Space $x$", labelpad=10)
# ax.set_ylabel(r"Time $t$", labelpad=10)
# ax.set_zlabel(r"Absolute error", labelpad=10)

# ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# ax.ticklabel_format(axis='z', style='sci', scilimits=(0,0))

# ax.view_init(elev=28, azim=225)
# ax.dist = 11
# ax.grid(False)

# plt.tight_layout()
# plt.savefig("Error.png", dpi=300, bbox_inches="tight")
# plt.savefig("Error.pdf", bbox_inches="tight")
# plt.show()

# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)

# end=datetime.now()
# print(end-start)


#New plot codding
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.ticker import ScalarFormatter
# from datetime import datetime
# import mpmath as mp

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120,
# })

# # --------------------------------------------------
# # mpmath -> numpy float conversion
# # --------------------------------------------------
# def to_float(a, vector=False):
#     """Convert mp.matrix / list of mpf / nested lists to a float ndarray."""
#     if isinstance(a, mp.matrix):
#         a = a.tolist()
#     arr = np.array([[float(v) for v in row] for row in a]) \
#         if (len(a) and isinstance(a[0], (list, tuple))) \
#         else np.array([float(v) for v in a])
#     return arr.ravel() if vector else arr

# X_f = to_float(X, vector=True)
# t_f = to_float(t, vector=True)

# Xplot_coarse, Yplot_coarse = np.meshgrid(X_f, t_f)   # shape (len(t), len(X))

# U_exact_f = to_float(U_exact)   # expected shape (len(X), len(t))
# U_f       = to_float(U)
# U_err_f   = to_float(U_error)

# # --------------------------------------------------
# # Generic surface plot
# # --------------------------------------------------
# def surface(Z, zlabel, fname, sci=False, animate=False):
#     fig = plt.figure(figsize=(7.5, 5.5))
#     ax = fig.add_subplot(111, projection="3d")

#     surf = ax.plot_surface(Xplot_coarse, Yplot_coarse, Z.T,
#                            cmap="viridis", linewidth=0,
#                            antialiased=True, shade=True)
#     surf.set_rasterized(True)

#     ax.set_xlabel(r"Space $x$", labelpad=10)
#     ax.set_ylabel(r"Time $t$", labelpad=10)
#     ax.set_zlabel(zlabel, labelpad=10)
#     if sci:
#         ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
#         ax.ticklabel_format(axis="z", style="sci", scilimits=(0, 0))

#     ax.view_init(elev=28, azim=225)
#     ax.set_box_aspect(None, zoom=0.9)   # replaces deprecated ax.dist
#     ax.grid(False)

#     plt.tight_layout()
#     plt.savefig(f"{fname}.png", dpi=300, bbox_inches="tight")
#     plt.savefig(f"{fname}.pdf", bbox_inches="tight")
#     plt.show()

#     if animate:
#         for angle in range(0, 360, 5):
#             ax.view_init(elev=30, azim=angle)
#             plt.draw()
#             plt.pause(0.1)

# surface(U_exact_f, r"Exact solution",     "exact_solution_surface_alpha06")
# surface(U_f,       r"Numerical solution", "numerical_solution_surface_alpha06")
# surface(U_err_f,   r"Absolute error",     "Error", sci=True)

# end = datetime.now()
# print(end - start)











Example_2 code.

In [ ]:
# THIS CODE IS WORKING FOR TIME AND GIVING RESULT 3-a

import matplotlib.pyplot as plt
import numpy as np
import math
#from scipy.linalg import solve
from datetime import datetime
start = datetime.now()
from scipy.linalg import solve_banded
from mpmath import mp
#from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import ScalarFormatter

mp.dps = 120;

a = mp.mpf("0.80")
a_s = mp.mpf("0.40")

g1 = mp.gamma(2 - a)
g1_s = mp.gamma(2 - a_s)
g = mp.gamma(3 - a)
g_s = mp.gamma(3 - a_s)

def f(x, t):
    # Parameters
    alpha0 = a
    alpha1 = a_s
    x = mp.mpf(x)
    t = mp.mpf(t)
    # Spatial part
    S = mp.sin((mp.pi/2)*x - mp.pi/2)
    # ============================================================
    # 1. Caputo derivative of order alpha0
    # ============================================================

    D_alpha0 = (
        mp.gamma(alpha0 + 1)
        + mp.gamma(2*alpha0 - alpha1 + 1)
          / mp.gamma(alpha0 - alpha1 + 1)
          * t**(alpha0 - alpha1)
        + mp.gamma(3*alpha0 - alpha1 + 1)
          / mp.gamma(2*alpha0 - alpha1 + 1)
          * t**(2*alpha0 - alpha1)
    ) * S

    # ============================================================
    # 2. Caputo derivative of order alpha1
    # ============================================================

    D_alpha1 = (
        mp.gamma(alpha0 + 1)
        / mp.gamma(alpha0 - alpha1 + 1)
        * t**(alpha0 - alpha1)

        + mp.gamma(2*alpha0 - alpha1 + 1)
        / mp.gamma(2*alpha0 - 2*alpha1 + 1)
        * t**(2*alpha0 - 2*alpha1)

        + mp.gamma(3*alpha0 - alpha1 + 1)
        / mp.gamma(3*alpha0 - 2*alpha1 + 1)
        * t**(3*alpha0 - 2*alpha1)
    ) * S


    # ============================================================
    # 3. Second derivative with respect to x
    # ============================================================

    F = (
        t**alpha0
        + t**(2*alpha0 - alpha1)
        + t**(3*alpha0 - alpha1)
    )

    D_xx = -(mp.pi**2 / 4) * F * S

    return D_alpha0+D_alpha1-D_xx

a_ = mp.mpf('1')
c_ = mp.mpf('0')
L  = mp.mpf('2')
T  = mp.mpf('1')

M=8*2
#N=math.floor(T*((M/L)**(6/(3-a)))))
N=200*2
h = mp.mpf(L) / mp.mpf(M)
r=mp.mpf(3/a)-1
#r=mp.mpf(1)

#t=[T*((i/N)**r) for i in range(0,N+1)]
#t = T * (np.linspace(0, 1, N+1) ** r)
t = [T * (mp.mpf(i) / N) ** r for i in range(N + 1)]

#N1=math.ceil((mp.power(N,a-3)/t[1]**(2-a))**(1/(a-2)))
N1 = int(mp.ceil(
    mp.power(
        mp.power(N, a - 3) / mp.power(t[1], 2 - a),
        1 / (a - 2)
    )
))

#tF=[t[1]*((i/N1)**r) for i in range(0,N1+1)]
#tF = t[1] * np.linspace(0, 1, N1+1) ** r
tF = [t[1] * mp.power(mp.mpf(i) / N1, r) for i in range(N1 + 1)]

# Tau=np.zeros(N+1)
# for i in range(1,N+1):
#     Tau[i]=t[i]-t[i-1]
#Tau = np.diff(t, prepend=0)  # prepend=0  t[−1] = 0
Tau = [mp.mpf('0')] + [t[i] - t[i-1] for i in range(1, len(t))]

#x=[i*h for i in range(0,M+1)]
# x = np.linspace(0, L, M+1)
# X=np.zeros(M+3)

L = mp.mpf('1') - mp.mpf('-1')
h = L / M

x = [-1 + mp.mpf(i) * h for i in range(M + 1)]

X = [mp.mpf('0') for _ in range(M+3)]

X[0]=x[0]
X[1]=(x[0]+x[1])/2
for j in range(2,M+1):
    X[j]=x[j-1]
X[M+1]=(x[M]+x[M-1])/2
X[M+2]=x[M]

# u=np.zeros((M+3,N1+1))
# U=np.zeros((M+3,N+1))
# exact=np.zeros((M+3,N+1))
# error=np.zeros((M+3,N+1))

# A1 = np.zeros((M+5, M+5))
# B1=np.zeros((M+5,M+5))
# B=np.zeros((M+5,M+5))
# C=np.zeros((N+1,M+5))
# C1=np.zeros((N+1,M+5))
# C2=np.zeros((N+1,M+5))
# W1=np.zeros(M+5)
# W=np.zeros(M+5)

u     = mp.matrix(M+3, N1+1)
U     = mp.matrix(M+3, N+1)
exact = mp.matrix(M+3, N+1)
error = mp.matrix(M+3, N+1)

A1 = mp.matrix(M+5, M+5)
B1 = mp.matrix(M+5, M+5)
B  = mp.matrix(M+5, M+5)

C  = mp.matrix(N+1, M+5)
C1 = mp.matrix(N+1, M+5)
C2 = mp.matrix(N+1, M+5)

W1 = mp.matrix(M+5, 1)
W  = mp.matrix(M+5, 1)

zeta   = mp.matrix(N1+1, N1+1)
zeta_s = mp.matrix(N1+1, N1+1)

c1 = mp.mpf(69120) * h**2 / mp.mpf(a_)
d1 = mp.mpf(4320) * h**2 / mp.mpf(a_)

# --- Compute zeta ---
# for n in range(1, N1 + 1):
#     dt = tF[n] - tF[:n]
#     dt1 = tF[n] - tF[1:n + 1]
#     zeta[:n, n] = (dt**(1 - a) - dt1**(1 - a)) / ((tF[1:n + 1] - tF[:n]) * g1)

for n in range(0,N1+1):
    for k in range(0,n):
        zeta[k,n] = ((mp.power(tF[n] - tF[k], 1-a) - mp.power(tF[n] - tF[k+1], 1-a)) /
                     ((tF[k+1] - tF[k]) * g1))

for n in range(0,N1+1):
    for k in range(0,n):
        zeta_s[k,n] = ((mp.power(tF[n] - tF[k], 1-a_s) - mp.power(tF[n] - tF[k+1], 1-a_s)) /
                     ((tF[k+1] - tF[k]) * g1_s))

for i in range(0,M+3):
    u[i,0]= 0
    U[i,0]=0


A1[0,0] = mp.mpf(1)/120   # first boundary condition
A1[0,1] = mp.mpf(26)/120
A1[0,2] = mp.mpf(66)/120
A1[0,3] = mp.mpf(26)/120
A1[0,4] = mp.mpf(1)/120

for i in range(M-2,M+3):
    A1[M+4,i+2]=A1[0,i-(M-2)]

A1[1,0]=-mp.mpf(717)/d1      # Equation with node x_0  # y-y"(0)
A1[1,1]=-mp.mpf(1448)/d1
A1[1,2]=mp.mpf(4300)/d1
A1[1,3]=-mp.mpf(1322)/d1
A1[1,4]=-mp.mpf(938)/d1
A1[1,5]=mp.mpf(202)/d1
A1[1,6]=-mp.mpf(92)/d1
A1[1,7]=mp.mpf(10)/d1
A1[1,8]=mp.mpf(7)/d1
A1[1,9]=-mp.mpf(2)/d1

for i in range(-7+M,M+3):
    A1[M+3,i+2]=A1[1,9-(i-(-7+M))]

A1[2,0]=-mp.mpf(1475)/c1         # Tau1
A1[2,1]=-mp.mpf(30149)/c1
A1[2,2]=mp.mpf(31918)/c1
A1[2,3]=mp.mpf(30322)/c1
A1[2,4]=-mp.mpf(27776)/c1
A1[2,5]=-mp.mpf(3680)/c1
A1[2,6]=mp.mpf(994)/c1
A1[2,7]=-mp.mpf(98)/c1
A1[2,8]=-mp.mpf(77)/c1
A1[2,9]=mp.mpf(21)/c1

for i in range(-7+M,M+3):
    A1[M+2,i+2]=A1[2,9-(i-(-7+M))]

A1[3,0]=mp.mpf(2)/d1                                # y-y"(1)
A1[3,1]=-mp.mpf(725)/d1
A1[3,2]=-mp.mpf(1454)/d1
A1[3,3]=mp.mpf(4396)/d1
A1[3,4]=-mp.mpf(1574)/d1
A1[3,5]=-mp.mpf(602)/d1
A1[3,6]=-mp.mpf(50)/d1
A1[3,7]=mp.mpf(4)/d1
A1[3,8]=mp.mpf(4)/d1
A1[3,9]=-mp.mpf(1)/d1

for i in range(-7+M,M+3):
    A1[M+1,i+2]=A1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    A1[i+2,-4+i+2]=mp.mpf(1)/d1
    A1[i+2,-3+i+2]=-mp.mpf(2)/d1
    A1[i+2,-2+i+2]=-mp.mpf(728)/d1
    A1[i+2,-1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,i+2]=mp.mpf(4270)/d1
    A1[i+2,1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,2+i+2]=-mp.mpf(728)/d1
    A1[i+2,3+i+2]=-mp.mpf(2)/d1
    A1[i+2,4+i+2]=mp.mpf(1)/d1

#------------------------------------------------------------------------------
# time level of first interval
B1[0,0]=A1[0,0]         # first boundary condition
B1[0,1]=A1[0,1]
B1[0,2]=A1[0,2]
B1[0,3]=A1[0,3]
B1[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B1[M+4,i+2]=B1[0,i-(M-2)]

# B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B1[1,1]=A1[1,1]+26*q1
# B1[1,2]=A1[1,2]+66*q1
# B1[1,3]=A1[1,3]+26*q1
# B1[1,4]=A1[1,4]+q1
B1[1,5]=A1[1,5]
B1[1,6]=A1[1,6]
B1[1,7]=A1[1,7]
B1[1,8]=A1[1,8]
B1[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

# B1[2,0]=A1[2,0]+q2         # Tau1
# B1[2,1]=A1[2,1]+237*q2
# B1[2,2]=A1[2,2]+1682*q2
# B1[2,3]=A1[2,3]+1682*q2
# B1[2,4]=A1[2,4]+237*q2
# B1[2,5]=A1[2,5]+q2
B1[2,6]=A1[2,6]
B1[2,7]=A1[2,7]
B1[2,8]=A1[2,8]
B1[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

B1[3,0]=A1[3,0]                                 # y-y"(1)
# B1[3,1]=A1[3,1]+q1
# B1[3,2]=A1[3,2]+26*q1
# B1[3,3]=A1[3,3]+66*q1
# B1[3,4]=A1[3,4]+26*q1
# B1[3,5]=A1[3,5]+q1
B1[3,6]=A1[3,6]
B1[3,7]=A1[3,7]
B1[3,8]=A1[3,8]
B1[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B1[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B1[i+2,3+i+2]=A1[i+2,3+i+2]
    B1[i+2,4+i+2]=A1[i+2,4+i+2]

for n in range(1,N1+1):
    q1=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(120)
    q2=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(3840)

    # B1[0,0]=A1[0,0]         # first boundary condition
    # B1[0,1]=A1[0,1]
    # B1[0,2]=A1[0,2]
    # B1[0,3]=A1[0,3]
    # B1[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B1[M+4,i+2]=B1[0,i-(M-2)]

    B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B1[1,1]=A1[1,1]+26*q1
    B1[1,2]=A1[1,2]+66*q1
    B1[1,3]=A1[1,3]+26*q1
    B1[1,4]=A1[1,4]+q1
    # B1[1,5]=A1[1,5]
    # B1[1,6]=A1[1,6]
    # B1[1,7]=A1[1,7]
    # B1[1,8]=A1[1,8]
    # B1[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

    B1[2,0]=A1[2,0]+q2         # Tau1
    B1[2,1]=A1[2,1]+237*q2
    B1[2,2]=A1[2,2]+1682*q2
    B1[2,3]=A1[2,3]+1682*q2
    B1[2,4]=A1[2,4]+237*q2
    B1[2,5]=A1[2,5]+q2
    # B1[2,6]=A1[2,6]
    # B1[2,7]=A1[2,7]
    # B1[2,8]=A1[2,8]
    # B1[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

    #B1[3,0]=A1[3,0]                                 # y-y"(1)
    B1[3,1]=A1[3,1]+q1
    B1[3,2]=A1[3,2]+26*q1
    B1[3,3]=A1[3,3]+66*q1
    B1[3,4]=A1[3,4]+26*q1
    B1[3,5]=A1[3,5]+q1
    # B1[3,6]=A1[3,6]
    # B1[3,7]=A1[3,7]
    # B1[3,8]=A1[3,8]
    # B1[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

    for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
        # B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B1[i+2,i+2]=A1[i+2,i+2]+66*q1
        B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B1[i+2,3+i+2]=A1[i+2,3+i+2]
        # B1[i+2,4+i+2]=A1[i+2,4+i+2]

    W1[0]= 0
    for i in range(0,M+3):
        s=0
        if n>1:
            s=mp.fdot(u[i,1:n]-u[i,0:n-1],zeta[0:n-1,n]+zeta_s[0:n-1,n])
        W1[i+1]=f(X[i],tF[n])+u[i,n-1]*(zeta[n-1,n]+zeta_s[n-1,n])-s
    W1[M+4]= 0

    # ab = extract_banded_form(B1, lower_bw=8, upper_bw=8)
    # G1 = solve_banded((8, 8), ab, W1)

    # for i in range(0,M+5):
    #     C1[n,i]=G1.item(i)

    B1_mp = mp.matrix(B1)
    W1_mp = mp.matrix(W1)

    G1 = mp.lu_solve(B1_mp, W1_mp)

    for i in range(M+5):
        C1[n, i] = G1[i]

    u[0,n]=(C1[n,0]+C1[n,1]*26+C1[n,2]*66+C1[n,3]*26+C1[n,4])/120
    u[1,n]=(C1[n,0]+C1[n,1]*237+C1[n,2]*1682+C1[n,3]*1682+C1[n,4]*237+C1[n,5])/3840

    for i in range(2,M+1):
        u[i,n]=(C1[n,i-3+2]+C1[n,i-2+2]*26+C1[n,i-1+2]*66+C1[n,i+2]*26+C1[n,i+1+2])/120

    u[M+1,n]=(C1[n,M-3+2]+237*C1[n,M-2+2]+1682*C1[n,M-1+2]+1682*C1[n,M+2]+237*C1[n,M+1+2]+C1[n,M+2+2])/3840
    u[M+2,n]=(C1[n,M-2+2]+C1[n,M-1+2]*26+C1[n,M+2]*66+C1[n,M+1+2]*26+C1[n,M+2+2])/120

#------------------------------------------------------------------------------------------------------
for i in range(0,M+3):
    U[i,1]=u[i,N1]

#Xi=np.zeros((N1+1,N+1))

# Xi = mp.matrix(N1+1, N+1)
# Xi_s = mp.matrix(N1+1, N+1)

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi[k,j]=(((t[j]-tF[k])**(1-a))-((t[j]-tF[k+1])**(1-a)))/((tF[k+1]-tF[k])*g1)

# #Xi_s=np.zeros((N1+1,N+1))

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi_s[k,j]=(((t[j]-tF[k])**(1-a_s))-((t[j]-tF[k+1])**(1-a_s)))/((tF[k+1]-tF[k])*g1_s)

Xi   = mp.matrix(N1 + 1, N + 1)
Xi_s = mp.matrix(N1 + 1, N + 1)

for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi[k, j] = (
            dt1**(1 - a) - dt2**(1 - a)
        ) / (hF * g1)


for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi_s[k, j] = (
            dt1**(1 - a_s) - dt2**(1 - a_s)
        ) / (hF * g1_s)


def gamma_mat(k,j):
    return ((mp.power(t[j] - t[k], 1 - a) - mp.power(t[j] - t[k+1], 1 - a)) / g1)

def gamma_mat_s(k,j):
    return ((mp.power(t[j] - t[k], 1 - a_s) - mp.power(t[j] - t[k+1], 1 - a_s)) / g1_s)

#Del=np.zeros((N+1,N+1))

#for j in range(0,N+1):
    #for k in range(0,j):
        #Del[k,j]=((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

#def Del(k,j):
    #return ((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

def Del(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a))
    term2 = mp.power(t[j] - t[k+1], 1 - a) * (t[j] + (a - 2) * t[k] + (1 - a) * t[k+1])
    return (term1 - term2) / g - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat(k, j)

def Del_s(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a_s))
    term2 = mp.power(t[j] - t[k+1], 1 - a_s) * (t[j] + (a_s - 2) * t[k] + (1 - a_s) * t[k+1])
    return (term1 - term2) / g_s - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat_s(k, j)

def d(k,j):
    if k == j:
        return (
            2 * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

def d_s(k,j):
    if k == j:
        return (
            2 * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat_s(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

B[0,0]=A1[0,0]         # first Boundary condition
B[0,1]=A1[0,1]
B[0,2]=A1[0,2]
B[0,3]=A1[0,3]
B[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B[M+4,i+2]=B[0,i-(M-2)]

# B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B[1,1]=A1[1,1]+26*q1
# B[1,2]=A1[1,2]+66*q1
# B[1,3]=A1[1,3]+26*q1
# B[1,4]=A1[1,4]+q1
B[1,5]=A1[1,5]
B[1,6]=A1[1,6]
B[1,7]=A1[1,7]
B[1,8]=A1[1,8]
B[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B[M+3,i+2]=B[1,9-(i-(-7+M))]

# B[2,0]=A1[2,0]+q2         # Tau1
# B[2,1]=A1[2,1]+237*q2
# B[2,2]=A1[2,2]+1682*q2
# B[2,3]=A1[2,3]+1682*q2
# B[2,4]=A1[2,4]+237*q2
# B[2,5]=A1[2,5]+q2
B[2,6]=A1[2,6]
B[2,7]=A1[2,7]
B[2,8]=A1[2,8]
B[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B[M+2,i+2]=B[2,9-(i-(-7+M))]

B[3,0]=A1[3,0]                                 # y-y"(1)
# B[3,1]=A1[3,1]+q1
# B[3,2]=A1[3,2]+26*q1
# B[3,3]=A1[3,3]+66*q1
# B[3,4]=A1[3,4]+26*q1
# B[3,5]=A1[3,5]+q1
B[3,6]=A1[3,6]
B[3,7]=A1[3,7]
B[3,8]=A1[3,8]
B[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B[i+2,3+i+2]=A1[i+2,3+i+2]
    B[i+2,4+i+2]=A1[i+2,4+i+2]


n=2
# --- D1 matrix ---
D1 = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1[0,0] = -two * mp.mpf(Del(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
D1[1,0] = two * Del(1, j) / (Tau[1] * Tau[2])
D1[2,0] = D1[0,0]+D1[1,0]

# --- D1 matrix ---
D1_s = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1_s[0,0] = -two * mp.mpf(Del_s(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
D1_s[1,0] = two * Del_s(1, j) / (Tau[1] * Tau[2])
D1_s[2,0] = D1_s[0,0]+D1_s[1,0]

q1=((D1[2,0]+D1_s[2,0])-c_)/120
q2=((D1[2,0]+D1_s[2,0])-c_)/3840

# B[0,0]=A1[0,0]         # first Boundary condition
# B[0,1]=A1[0,1]
# B[0,2]=A1[0,2]
# B[0,3]=A1[0,3]
# B[0,4]=A1[0,4]

# for i in range(M-2,M+3):
#     B[M+4,i+2]=B[0,i-(M-2)]

B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
B[1,1]=A1[1,1]+26*q1
B[1,2]=A1[1,2]+66*q1
B[1,3]=A1[1,3]+26*q1
B[1,4]=A1[1,4]+q1
# B[1,5]=A1[1,5]
# B[1,6]=A1[1,6]
# B[1,7]=A1[1,7]
# B[1,8]=A1[1,8]
# B[1,9]=A1[1,9]

for i in range(-7+M,M+3):
    B[M+3,i+2]=B[1,9-(i-(-7+M))]

B[2,0]=A1[2,0]+q2         # Tau1
B[2,1]=A1[2,1]+237*q2
B[2,2]=A1[2,2]+1682*q2
B[2,3]=A1[2,3]+1682*q2
B[2,4]=A1[2,4]+237*q2
B[2,5]=A1[2,5]+q2
# B[2,6]=A1[2,6]
# B[2,7]=A1[2,7]
# B[2,8]=A1[2,8]
# B[2,9]=A1[2,9]

for i in range(-7+M,M+3):
    B[M+2,i+2]=B[2,9-(i-(-7+M))]

# B[3,0]=A1[3,0]                                 # y-y"(1)
B[3,1]=A1[3,1]+q1
B[3,2]=A1[3,2]+26*q1
B[3,3]=A1[3,3]+66*q1
B[3,4]=A1[3,4]+26*q1
B[3,5]=A1[3,5]+q1
# B[3,6]=A1[3,6]
# B[3,7]=A1[3,7]
# B[3,8]=A1[3,8]
# B[3,9]=A1[3,9]

for i in range(-7+M,M+3):
    B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
    # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    B[i+2,i+2]=A1[i+2,i+2]+66*q1
    B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    # B[i+2,3+i+2]=A1[i+2,3+i+2]
    # B[i+2,4+i+2]=A1[i+2,4+i+2]

W[0]=0
for i in range(0,M+3):
    W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot(D1[0:n,0]+D1_s[0:n,0],U[i,0:n])+f(X[i],t[n])
W[M+4]=0

# ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
# G = solve_banded((8, 8), ab, W)

# for i in range(0,M+5):
#     C2[n,i]=G.item(i)

B_mp = mp.matrix(B)
W_mp = mp.matrix(W)

G = mp.lu_solve(B_mp, W_mp)

for i in range(M+5):
     C2[n, i] = G[i]

U[0,n]=(C2[n,0]+C2[n,1]*26+C2[n,2]*66+C2[n,3]*26+C2[n,4])/120
U[1,n]=(C2[n,0]+C2[n,1]*237+C2[n,2]*1682+C2[n,3]*1682+C2[n,4]*237+C2[n,5])/3840

for i in range(2,M+1):
    U[i,n]=(C2[n,i-3+2]+C2[n,i-2+2]*26+C2[n,i-1+2]*66+C2[n,i+2]*26+C2[n,i+1+2])/120

U[M+1,n]=(C2[n,M-3+2]+237*C2[n,M-2+2]+1682*C2[n,M-1+2]+1682*C2[n,M+2]+237*C2[n,M+1+2]+C2[n,M+2+2])/3840
U[M+2,n]=(C2[n,M-2+2]+C2[n,M-1+2]*26+C2[n,M+2]*66+C2[n,M+1+2]*26+C2[n,M+2+2])/120


for n in range(3,N+1):
    # --- D1 matrix ---
    D1 = mp.zeros(N+1, 1)  # ✅ allows mpf/mpc objects
    two=mp.mpf(2)
    j=n
    D1[0,0] = -two * Del(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1[1,0] = (
        -two * Del(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat(2, j) / (Tau[2] + Tau[3])
    )

    if j>3:
        for k in range(2, j - 1):
            D1[k,0] = (
                -two * Del(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
                + two * Del(k, j) / (Tau[k] * Tau[k + 1])
                - two * Del(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
                - gamma_mat(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
                + gamma_mat(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
            )

    D1[j - 1,0] = (
        -two * Del(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1[j,0] = (
        two * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    # --- D1 matrix ---
    D1_s = mp.zeros(N+1,1)
    two=mp.mpf(2)
    j=n
    D1_s[0,0] = -two * Del_s(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1_s[1,0] = (
        -two * Del_s(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del_s(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat_s(2, j) / (Tau[2] + Tau[3])
    )
    for k in range(2, j - 1):
        D1_s[k,0] = (
            -two * Del_s(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
            + two * Del_s(k, j) / (Tau[k] * Tau[k + 1])
            - two * Del_s(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
            - gamma_mat_s(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
            + gamma_mat_s(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
        )
    D1_s[j - 1,0] = (
        -two * Del_s(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del_s(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat_s(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1_s[j,0] = (
        two * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat_s(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    q1=((d(n,n)+d_s(n,n))-c_)/120
    q2=((d(n,n)+d_s(n,n))-c_)/3840

    # B[0,0]=A1[0,0]         # first Boundary condition
    # B[0,1]=A1[0,1]
    # B[0,2]=A1[0,2]
    # B[0,3]=A1[0,3]
    # B[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B[M+4,i+2]=B[0,i-(M-2)]

    B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1
    # B[1,5]=A1[1,5]
    # B[1,6]=A1[1,6]
    # B[1,7]=A1[1,7]
    # B[1,8]=A1[1,8]
    # B[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2
    # B[2,6]=A1[2,6]
    # B[2,7]=A1[2,7]
    # B[2,8]=A1[2,8]
    # B[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    # B[3,0]=A1[3,0]                                 # y-y"(1)
    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1
    # B[3,6]=A1[3,6]
    # B[3,7]=A1[3,7]
    # B[3,8]=A1[3,8]
    # B[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
        # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B[i+2,3+i+2]=A1[i+2,3+i+2]
        # B[i+2,4+i+2]=A1[i+2,4+i+2]

    W[0]=0
    for i in range(0,M+3):
        W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot((D1[0:n,0]+D1_s[0:n,0]),U[i,0:n])+f(X[i],t[n])
    W[M+4]=0

    # ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
    # G = solve_banded((8, 8), ab, W)

    # for i in range(0,M+5):
    #     C[n,i]=G.item(i)

    B_mp = mp.matrix(B)
    W_mp = mp.matrix(W)

    G = mp.lu_solve(B_mp, W_mp)

    for i in range(M+5):
         C[n, i] = G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

    U[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
    U[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

# u_exact=np.zeros((M+3,N1+1))
# u_error=np.zeros((M+3,N1+1))
# for j in range(0,N1+1):
#     for i in range(0,M+3):
#         u_exact[i,j]=(tF[j]**2)*(mp.sin(2*mp.pi*X[i]))
#         u_error[i,j]=abs(u_exact[i,j]-u[i,j])

# U_exact=np.zeros((M+3,N+1))
# U_error=np.zeros((M+3,N+1))
U_exact = mp.matrix(M+3, N+1)
U_error = mp.matrix(M+3, N+1)

for i in range(0,M+3):
    for j in range(0,N+1):
        U_exact[i,j]=(t[j]**a+t[j]**(2*a-a_s)+t[j]**(3*a-a_s))*(mp.sin(mp.pi*(0.5)*X[i]-mp.pi*(0.5)))
        U_error[i,j]=abs(U_exact[i,j]-U[i,j])

#print(np.max(U_error))
max_error = max(abs(U_error[i,j]) for i in range(U_error.rows) for j in range(U_error.cols))
print(max_error)

# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Meshgrid (coarse grid)
# # --------------------------------------------------
# Xplot_coarse, Yplot_coarse = np.meshgrid(X, t)

# Xplot_coarse = Xplot_coarse.astype(float)
# Yplot_coarse = Yplot_coarse.astype(float)
# V_float1 = np.array(U_exact, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float1.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Exact solution")
# ax.grid(False)

# # Fixed good viewing angle (for paper)
# ax.view_init(elev=28, azim=225)


# # --------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("exact_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("exact_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -------------------------------------------------------------------------------------------------------------------------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Convert numerical solution to float
# # --------------------------------------------------
# V_float = np.array(U, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Numerical solution ")
# ax.grid(False)

# # Fixed good viewing angle (match exact solution plot)
# ax.view_init(elev=28, azim=225)

# # -------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("numerical_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("numerical_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "lines.linewidth": 2,
#     "figure.dpi": 120
# })

# #-------------------------------------------------------------------------------------------------------------------------------------

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# U_error_float = np.array(U_error, dtype=float)

# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, U_error_float.T,
#     cmap='viridis',
#     linewidth=0,
#     antialiased=True,
#     shade=True
# )

# surf.set_rasterized(True)

# ax.set_xlabel(r"Space $x$", labelpad=10)
# ax.set_ylabel(r"Time $t$", labelpad=10)
# ax.set_zlabel(r"Absolute error", labelpad=10)

# ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# ax.ticklabel_format(axis='z', style='sci', scilimits=(0,0))

# ax.view_init(elev=28, azim=225)
# ax.dist = 11
# ax.grid(False)

# plt.tight_layout()
# plt.savefig("Error.png", dpi=300, bbox_inches="tight")
# plt.savefig("Error.pdf", bbox_inches="tight")
# plt.show()

# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)

# end=datetime.now()
# print(end-start)

# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.ticker import ScalarFormatter
# from datetime import datetime
# import mpmath as mp

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120,
# })

# # --------------------------------------------------
# # mpmath -> numpy float conversion
# # --------------------------------------------------
# def to_float(a, vector=False):
#     """Convert mp.matrix / list of mpf / nested lists to a float ndarray."""
#     if isinstance(a, mp.matrix):
#         a = a.tolist()
#     arr = np.array([[float(v) for v in row] for row in a]) \
#         if (len(a) and isinstance(a[0], (list, tuple))) \
#         else np.array([float(v) for v in a])
#     return arr.ravel() if vector else arr

# X_f = to_float(X, vector=True)
# t_f = to_float(t, vector=True)

# Xplot_coarse, Yplot_coarse = np.meshgrid(X_f, t_f)   # shape (len(t), len(X))

# U_exact_f = to_float(U_exact)   # expected shape (len(X), len(t))
# U_f       = to_float(U)
# U_err_f   = to_float(U_error)

# # --------------------------------------------------
# # Generic surface plot
# # --------------------------------------------------
# def surface(Z, zlabel, fname, sci=False, animate=False):
#     fig = plt.figure(figsize=(7.5, 5.5))
#     ax = fig.add_subplot(111, projection="3d")

#     surf = ax.plot_surface(Xplot_coarse, Yplot_coarse, Z.T,
#                            cmap="viridis", linewidth=0,
#                            antialiased=True, shade=True)
#     surf.set_rasterized(True)

#     ax.set_xlabel(r"Space $x$", labelpad=10)
#     ax.set_ylabel(r"Time $t$", labelpad=10)
#     ax.set_zlabel(zlabel, labelpad=10)
#     if sci:
#         ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
#         ax.ticklabel_format(axis="z", style="sci", scilimits=(0, 0))

#     ax.view_init(elev=28, azim=225)
#     ax.set_box_aspect(None, zoom=0.9)   # replaces deprecated ax.dist
#     ax.grid(False)

#     plt.tight_layout()
#     plt.savefig(f"{fname}.png", dpi=300, bbox_inches="tight")
#     plt.savefig(f"{fname}.pdf", bbox_inches="tight")
#     plt.show()

#     if animate:
#         for angle in range(0, 360, 5):
#             ax.view_init(elev=30, azim=angle)
#             plt.draw()
#             plt.pause(0.1)

# surface(U_exact_f, r"Exact solution",     "exact_solution_surface_alpha06")
# surface(U_f,       r"Numerical solution", "numerical_solution_surface_alpha06")
# surface(U_err_f,   r"Absolute error",     "Error", sci=True)

# end = datetime.now()
# print(end - start)














Example_3 code.

In [ ]:
import sys
from operator import mul
from datetime import datetime
from mpmath import mp

start = datetime.now()

# ----------------------------------------------------------------------------
# Settings.  Optional command line:  python multiterm_fast_sin2.py M N DPS
# (defaults are your original M=16, N=400; dps 120 -> 40)
# ----------------------------------------------------------------------------
M   = int(sys.argv[1]) if len(sys.argv) > 1 else 8*2
N   = int(sys.argv[2]) if len(sys.argv) > 2 else 20000
DPS = int(sys.argv[3]) if len(sys.argv) > 3 else 50
mp.dps = DPS
mpf = mp.mpf

a_l = mpf("0.80")
a   = a_l
a_s = mpf("0.40")
g1, g1_s = mp.gamma(2 - a), mp.gamma(2 - a_s)
g,  g_s  = mp.gamma(3 - a), mp.gamma(3 - a_s)
ig1, ig1_s, ig, ig_s = 1/g1, 1/g1_s, 1/g, 1/g_s

a_, c_, T = mpf(1), mpf(0), mpf(1)
L = mpf(2)                      # space interval is [-1, 1]
h = L / M
r = 3 / a - 1
zero = mpf(0)

# ----------------------------------------------------------------------------
# PROBLEM DATA  (exact solution u = tfun(t) * Sfun(x);  f = ftime(t) * Sfun(x))
# ----------------------------------------------------------------------------
def tfun(t):
    return t**a + t**(2*a - a_s) + t**(3*a - a_s)

def Sfun(x):
    return mp.sin((mp.pi/2) * x - mp.pi/2)

def ftime(t):
    """f(x,t) = ftime(t) * Sfun(x)   (= D^a0 u + D^a1 u - u_xx)"""
    al0, al1 = a, a_s
    G = mp.gamma
    D0 = (G(al0 + 1)
          + G(2*al0 - al1 + 1) / G(al0 - al1 + 1) * t**(al0 - al1)
          + G(3*al0 - al1 + 1) / G(2*al0 - al1 + 1) * t**(2*al0 - al1))
    D1 = (G(al0 + 1) / G(al0 - al1 + 1) * t**(al0 - al1)
          + G(2*al0 - al1 + 1) / G(2*al0 - 2*al1 + 1) * t**(2*al0 - 2*al1)
          + G(3*al0 - al1 + 1) / G(3*al0 - 2*al1 + 1) * t**(3*al0 - 2*al1))
    F = t**al0 + t**(2*al0 - al1) + t**(3*al0 - al1)
    return D0 + D1 + (mp.pi**2 / 4) * F          # -D_xx = +(pi^2/4) F S

def bc_left(t):                    # u(-1,t) = 0
    return zero

def bc_right(t):                   # u(+1,t) = 0
    return zero

# ----------------------------------------------------------------------------
# Grids
# ----------------------------------------------------------------------------
t = [T * (mpf(i) / N) ** r for i in range(N + 1)]
N1 = int(mp.ceil(mp.power(mp.power(N, a - 3) / mp.power(t[1], 2 - a), 1 / (a - 2))))
tF = [t[1] * mp.power(mpf(i) / N1, r) for i in range(N1 + 1)]
Tau = [zero] + [t[i] - t[i-1] for i in range(1, N + 1)]

x = [-1 + mpf(i) * h for i in range(M + 1)]
X = [zero] * (M + 3)
X[0] = x[0]
X[1] = (x[0] + x[1]) / 2
for j in range(2, M + 1):
    X[j] = x[j-1]
X[M+1] = (x[M] + x[M-1]) / 2
X[M+2] = x[M]
S = [Sfun(xi) for xi in X]                 # spatial profile, never changes in time

# ----------------------------------------------------------------------------
# Matrix pieces (sparse rows, stored as dicts {col: value})
# ----------------------------------------------------------------------------
c1 = mpf(69120) * h**2 / a_
d1 = mpf(4320)  * h**2 / a_

S5 = [1, 26, 66, 26, 1]
S6 = [1, 237, 1682, 1682, 237, 1]
R0 = [mpf(v) / 120 for v in S5]
R1 = [mpf(v) / d1 for v in (-717, -1448, 4300, -1322, -938, 202, -92, 10, 7, -2)]
R2 = [mpf(v) / c1 for v in (-1475, -30149, 31918, 30322, -27776, -3680, 994, -98, -77, 21)]
R3 = [mpf(v) / d1 for v in (2, -725, -1454, 4396, -1574, -602, -50, 4, 4, -1)]
RI = [mpf(v) / d1 for v in (1, -2, -728, -1406, 4270, -1406, -728, -2, 1)]

def mirror(row):
    return {M + 4 - c: v for c, v in row.items()}

def build_rows(q1, q2):
    qa = [q1 * s for s in S5]
    qb = [q2 * s for s in S6]
    rows = [None] * (M + 5)
    rows[0] = dict(enumerate(R0))
    rows[M+4] = mirror(rows[0])
    r1 = dict(enumerate(R1))
    for k in range(5): r1[k] += qa[k]
    r2 = dict(enumerate(R2))
    for k in range(6): r2[k] += qb[k]
    r3 = dict(enumerate(R3))
    for k in range(5): r3[1 + k] += qa[k]
    rows[1], rows[2], rows[3] = r1, r2, r3
    rows[M+3], rows[M+2], rows[M+1] = mirror(r1), mirror(r2), mirror(r3)
    for i in range(4, M + 1):
        row = {i - 4 + k: RI[k] for k in range(9)}
        for k in range(5): row[i - 2 + k] += qa[k]
        rows[i] = row
    return rows

def solve_banded_mp(rows, rhs, kl=8):
    n = len(rows)
    b = list(rhs)
    for k in range(n):
        hi = min(n, k + kl + 1)
        p, best = k, abs(rows[k].get(k, 0))
        for rr in range(k + 1, hi):
            v = abs(rows[rr].get(k, 0))
            if v > best:
                best, p = v, rr
        if p != k:
            rows[k], rows[p] = rows[p], rows[k]
            b[k], b[p] = b[p], b[k]
        rk, piv, bk = rows[k], rows[k][k], b[k]
        for rr in range(k + 1, hi):
            row = rows[rr]
            v = row.pop(k, None)
            if not v:
                continue
            m = v / piv
            for c, val in rk.items():
                if c != k:
                    row[c] = row.get(c, 0) - m * val
            b[rr] -= m * bk
    xs = [zero] * n
    for i in range(n - 1, -1, -1):
        row = rows[i]
        s = b[i]
        for c, v in row.items():
            if c > i:
                s -= v * xs[c]
        xs[i] = s / row[i]
    return xs

def recover(c):
    v = [zero] * (M + 3)
    v[0] = (c[0] + 26*c[1] + 66*c[2] + 26*c[3] + c[4]) / 120
    v[1] = (c[0] + 237*c[1] + 1682*c[2] + 1682*c[3] + 237*c[4] + c[5]) / 3840
    for i in range(2, M + 1):
        v[i] = (c[i-1] + 26*c[i] + 66*c[i+1] + 26*c[i+2] + c[i+3]) / 120
    v[M+1] = (c[M-1] + 237*c[M] + 1682*c[M+1] + 1682*c[M+2] + 237*c[M+3] + c[M+4]) / 3840
    v[M+2] = (c[M] + 26*c[M+1] + 66*c[M+2] + 26*c[M+3] + c[M+4]) / 120
    return v

# ----------------------------------------------------------------------------
# Phase 1: fine graded mesh on [0, t1]
# ----------------------------------------------------------------------------
def zeta_vec(n):
    out = []
    for k in range(n):
        e1, e2 = tF[n] - tF[k], tF[n] - tF[k+1]
        hF = tF[k+1] - tF[k]
        out.append((e1**(1-a)   - e2**(1-a))   / (hF * g1)
                 + (e1**(1-a_s) - e2**(1-a_s)) / (hF * g1_s))
    return out

u = [[zero] for _ in range(M + 3)]
for n in range(1, N1 + 1):
    zc = zeta_vec(n)
    zl = zc[n-1]
    rows = build_rows((zl - c_) / 120, (zl - c_) / 3840)
    tn = tF[n]
    fc = ftime(tn)
    rhs = [zero] * (M + 5)
    rhs[0] = bc_left(tn)
    for i in range(M + 3):
        ui = u[i]
        s = mp.fsum((ui[k+1] - ui[k]) * zc[k] for k in range(n - 1)) if n > 1 else 0
        rhs[i+1] = fc * S[i] + ui[n-1] * zl - s
    rhs[M+4] = bc_right(tn)
    vals = recover(solve_banded_mp(rows, rhs))
    for i in range(M + 3):
        u[i].append(vals[i])

# ----------------------------------------------------------------------------
# Phase 2: coarse graded mesh on [t1, T]
# ----------------------------------------------------------------------------
hFs = [tF[k+1] - tF[k] for k in range(N1)]

def xi_vec(tj):
    out = []
    for k in range(N1):
        e1, e2 = tj - tF[k], tj - tF[k+1]
        out.append((e1**(1-a)   - e2**(1-a))   / (hFs[k] * g1)
                 + (e1**(1-a_s) - e2**(1-a_s)) / (hFs[k] * g1_s))
    return out

# ---- j-independent coefficient tables (computed once) ----------------------
hdT  = [(Tau[k+1] - Tau[k]) / 2 for k in range(N)]
tka  = [(a   - 2) * t[k] + (1 - a)   * t[k+1] for k in range(N)]
tkas = [(a_s - 2) * t[k] + (1 - a_s) * t[k+1] for k in range(N)]
CA = [zero] * (N + 3); CB = [zero] * (N + 3); CC = [zero] * (N + 3); CE = [zero] * (N + 3)
for k in range(1, N + 1):
    CE[k] = 1 / (Tau[k-1] + Tau[k])
for k in range(2, N - 1):
    CA[k] = 2 / (Tau[k] * (Tau[k-1] + Tau[k]))
    CB[k] = 2 / (Tau[k] * Tau[k+1])
    CC[k] = 2 / (Tau[k+1] * (Tau[k+1] + Tau[k+2]))

# If (1-a_s)/(1-a) is an integer m (here 0.6/0.2 = 3), then
# (t_j-t_k)^(1-a_s) = ((t_j-t_k)^(1-a))^m: one mp.power per k instead of two.
m_ratio = (1 - a_s) / (1 - a)
m_int = int(mp.nint(m_ratio))
USE_POWER_TRICK = abs(m_ratio - m_int) < mpf(10)**(-(DPS - 8)) and 1 <= m_int <= 8

def build_D(j):
    """Combined weights D[0..j] = D(alpha) + D_s(alpha_s) for time level j."""
    tj = t[j]
    dt = [tj - t[k] for k in range(j)]
    pw = [mp.power(d_, 1 - a) for d_ in dt] + [zero]
    if USE_POWER_TRICK:
        pws = [p_**m_int for p_ in pw]
    else:
        pws = [mp.power(d_, 1 - a_s) for d_ in dt] + [zero]
    Gm = [zero] * j
    Dl = [zero] * j
    for k in range(j):
        p0, p1 = pw[k], pw[k+1]
        q0, q1_ = pws[k], pws[k+1]
        gm_a  = (p0 - p1) * ig1
        gm_s  = (q0 - q1_) * ig1_s
        Gm[k] = gm_a + gm_s
        Dl[k] = ((p0 * dt[k] - p1 * (tj + tka[k])) * ig
                 + (q0 * dt[k] - q1_ * (tj + tkas[k])) * ig_s
                 - hdT[k] * Gm[k])
    Tn = Tau
    D = [zero] * (j + 1)
    D[0] = -2*Dl[1] / (Tn[1]*(Tn[1]+Tn[2])) + Gm[1] * CE[2]
    if j == 2:      # special first step used in the original code
        D[1] = 2*Dl[1] / (Tn[1]*Tn[2])
        D[2] = D[0] + D[1]
        return D
    D[1] = (-2*Dl[2] / (Tn[2]*(Tn[2]+Tn[3])) + 2*Dl[1] / (Tn[1]*Tn[2])
            + Gm[2] * CE[3])
    for k in range(2, j - 1):
        D[k] = (-CA[k]*Dl[k-1] + CB[k]*Dl[k] - CC[k]*Dl[k+1]
                - CE[k]*Gm[k-1] + CE[k+2]*Gm[k+1])
    D[j-1] = (-2*Dl[j-2] / (Tn[j-1]*(Tn[j-2]+Tn[j-1]))
              + 2*Dl[j-1] / (Tn[j-1]*Tn[j])
              - Gm[j-2] * CE[j-1])
    D[j] = 2*Dl[j-1] / (Tn[j]*(Tn[j-1]+Tn[j])) + Gm[j-1] * CE[j]
    return D

# ---- fixed-point storage of U: the O(N^2 * M) history sums become integer dots
P = int(3.33 * DPS) + 64
def to_fixed(v):
    s, man, ex_, bc = v._mpf_
    sh = ex_ + P
    val = man << sh if sh >= 0 else man >> (-sh)
    return -val if s else val
def from_fixed2(v):                   # value carrying scale 2^(2P)
    return mp.ldexp(mpf(v), -2 * P)

U = [[0, to_fixed(u[i][N1])] for i in range(M + 3)]     # integers, scale 2^P

for n in range(2, N + 1):
    Dsum = build_D(n)
    q1 = (Dsum[n] - c_) / 120
    q2 = (Dsum[n] - c_) / 3840
    rows = build_rows(q1, q2)

    Xc = xi_vec(t[n])
    wh = [zero] * (N1 + 1)
    wh[0] = Xc[0]
    for k in range(1, N1):
        wh[k] = Xc[k] - Xc[k-1]
    wh[N1] = -Xc[N1-1]

    tn = t[n]
    fc = ftime(tn)
    Dn = [to_fixed(d_) for d_ in Dsum[:n]]
    rhs = [zero] * (M + 5)
    rhs[0] = bc_left(tn)
    for i in range(M + 3):
        hist = from_fixed2(sum(map(mul, Dn, U[i])))     # len(U[i]) == n
        rhs[i+1] = mp.fdot(wh, u[i]) + hist + fc * S[i]
    rhs[M+4] = bc_right(tn)

    vals = recover(solve_banded_mp(rows, rhs))
    for i in range(M + 3):
        U[i].append(to_fixed(vals[i]))

# ----------------------------------------------------------------------------
# Error
# ----------------------------------------------------------------------------
tf = [tfun(tj) for tj in t]
max_error = max(abs(tf[j] * S[i] - mp.ldexp(mpf(U[i][j]), -P))
                for i in range(M + 3) for j in range(N + 1))
print("M =", M, " N =", N, " N1 =", N1, " dps =", DPS, " power trick:", USE_POWER_TRICK)
print("max error:", mp.nstr(max_error, 20))
print("time:", datetime.now() - start)


Example_3_new_code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
#from scipy.linalg import solve
from datetime import datetime
start = datetime.now()
from scipy.linalg import solve_banded
from mpmath import mp
#from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import ScalarFormatter

mp.dps = 120;

a_l=mp.mpf("0.80")
a = a_l
a_s = mp.mpf("0.40")

g1 = mp.gamma(2 - a)
g1_s = mp.gamma(2 - a_s)
g = mp.gamma(3 - a)
g_s = mp.gamma(3 - a_s)

def f(x, t):
    x = mp.mpf(x)
    t = mp.mpf(t)
    return (mp.power(t,a_l-a)*(mp.gamma(a_l+1)/mp.gamma(a_l-a+1))+mp.power(t,a_l-a_s)*(mp.gamma(a_l+1)/mp.gamma(a_l-a_s+1)))*mp.sin(2*mp.pi*x)+4*(mp.pi**2)*(t**a_l*mp.sin(2*mp.pi*x))

a_ = mp.mpf('1')
c_ = mp.mpf('0')
L  = mp.mpf('1')
T  = mp.mpf('1')


M=8*2*2*2
#N=math.floor(T*((M/L)**(6/(3-a))))
N=5000

h = mp.mpf(L) / mp.mpf(M)
r=mp.mpf(3/a)-1
#r=mp.mpf(1)

#t=[T*((i/N)**r) for i in range(0,N+1)]
#t = T * (np.linspace(0, 1, N+1) ** r)
t = [T * (mp.mpf(i) / N) ** r for i in range(N + 1)]

#N1=math.ceil((mp.power(N,a-3)/t[1]**(2-a))**(1/(a-2)))
N1 = int(mp.ceil(
    mp.power(
        mp.power(N, a - 3) / mp.power(t[1], 2 - a),
        1 / (a - 2)
    )
))

#tF=[t[1]*((i/N1)**r) for i in range(0,N1+1)]
#tF = t[1] * np.linspace(0, 1, N1+1) ** r
tF = [t[1] * mp.power(mp.mpf(i) / N1, r) for i in range(N1 + 1)]

# Tau=np.zeros(N+1)
# for i in range(1,N+1):
#     Tau[i]=t[i]-t[i-1]
#Tau = np.diff(t, prepend=0)  # prepend=0  t[−1] = 0
Tau = [mp.mpf('0')] + [t[i] - t[i-1] for i in range(1, len(t))]

#x=[i*h for i in range(0,M+1)]
# x = np.linspace(0, L, M+1)
# X=np.zeros(M+3)

x = [mp.mpf(i) * L / M for i in range(M+1)]
X = [mp.mpf('0') for _ in range(M+3)]

X[0]=x[0]
X[1]=(x[0]+x[1])/2
for j in range(2,M+1):
    X[j]=x[j-1]
X[M+1]=(x[M]+x[M-1])/2
X[M+2]=x[M]

# u=np.zeros((M+3,N1+1))
# U=np.zeros((M+3,N+1))
# exact=np.zeros((M+3,N+1))
# error=np.zeros((M+3,N+1))

# A1 = np.zeros((M+5, M+5))
# B1=np.zeros((M+5,M+5))
# B=np.zeros((M+5,M+5))
# C=np.zeros((N+1,M+5))
# C1=np.zeros((N+1,M+5))
# C2=np.zeros((N+1,M+5))
# W1=np.zeros(M+5)
# W=np.zeros(M+5)

u     = mp.matrix(M+3, N1+1)
U     = mp.matrix(M+3, N+1)
exact = mp.matrix(M+3, N+1)
error = mp.matrix(M+3, N+1)

A1 = mp.matrix(M+5, M+5)
B1 = mp.matrix(M+5, M+5)
B  = mp.matrix(M+5, M+5)

C  = mp.matrix(N+1, M+5)
C1 = mp.matrix(N+1, M+5)
C2 = mp.matrix(N+1, M+5)

W1 = mp.matrix(M+5, 1)
W  = mp.matrix(M+5, 1)

zeta   = mp.matrix(N1+1, N1+1)
zeta_s = mp.matrix(N1+1, N1+1)

c1 = mp.mpf(69120) * h**2 / mp.mpf(a_)
d1 = mp.mpf(4320) * h**2 / mp.mpf(a_)

# --- Compute zeta ---
# for n in range(1, N1 + 1):
#     dt = tF[n] - tF[:n]
#     dt1 = tF[n] - tF[1:n + 1]
#     zeta[:n, n] = (dt**(1 - a) - dt1**(1 - a)) / ((tF[1:n + 1] - tF[:n]) * g1)

for n in range(0,N1+1):
    for k in range(0,n):
        zeta[k,n] = ((mp.power(tF[n] - tF[k], 1-a) - mp.power(tF[n] - tF[k+1], 1-a)) /
                     ((tF[k+1] - tF[k]) * g1))

for n in range(0,N1+1):
    for k in range(0,n):
        zeta_s[k,n] = ((mp.power(tF[n] - tF[k], 1-a_s) - mp.power(tF[n] - tF[k+1], 1-a_s)) /
                     ((tF[k+1] - tF[k]) * g1_s))


A1[0,0] = mp.mpf(1)/120   # first boundary condition
A1[0,1] = mp.mpf(26)/120
A1[0,2] = mp.mpf(66)/120
A1[0,3] = mp.mpf(26)/120
A1[0,4] = mp.mpf(1)/120

for i in range(M-2,M+3):
    A1[M+4,i+2]=A1[0,i-(M-2)]

A1[1,0]=-mp.mpf(717)/d1      # Equation with node x_0  # y-y"(0)
A1[1,1]=-mp.mpf(1448)/d1
A1[1,2]=mp.mpf(4300)/d1
A1[1,3]=-mp.mpf(1322)/d1
A1[1,4]=-mp.mpf(938)/d1
A1[1,5]=mp.mpf(202)/d1
A1[1,6]=-mp.mpf(92)/d1
A1[1,7]=mp.mpf(10)/d1
A1[1,8]=mp.mpf(7)/d1
A1[1,9]=-mp.mpf(2)/d1

for i in range(-7+M,M+3):
    A1[M+3,i+2]=A1[1,9-(i-(-7+M))]

A1[2,0]=-mp.mpf(1475)/c1         # Tau1
A1[2,1]=-mp.mpf(30149)/c1
A1[2,2]=mp.mpf(31918)/c1
A1[2,3]=mp.mpf(30322)/c1
A1[2,4]=-mp.mpf(27776)/c1
A1[2,5]=-mp.mpf(3680)/c1
A1[2,6]=mp.mpf(994)/c1
A1[2,7]=-mp.mpf(98)/c1
A1[2,8]=-mp.mpf(77)/c1
A1[2,9]=mp.mpf(21)/c1

for i in range(-7+M,M+3):
    A1[M+2,i+2]=A1[2,9-(i-(-7+M))]

A1[3,0]=mp.mpf(2)/d1                                # y-y"(1)
A1[3,1]=-mp.mpf(725)/d1
A1[3,2]=-mp.mpf(1454)/d1
A1[3,3]=mp.mpf(4396)/d1
A1[3,4]=-mp.mpf(1574)/d1
A1[3,5]=-mp.mpf(602)/d1
A1[3,6]=-mp.mpf(50)/d1
A1[3,7]=mp.mpf(4)/d1
A1[3,8]=mp.mpf(4)/d1
A1[3,9]=-mp.mpf(1)/d1

for i in range(-7+M,M+3):
    A1[M+1,i+2]=A1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    A1[i+2,-4+i+2]=mp.mpf(1)/d1
    A1[i+2,-3+i+2]=-mp.mpf(2)/d1
    A1[i+2,-2+i+2]=-mp.mpf(728)/d1
    A1[i+2,-1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,i+2]=mp.mpf(4270)/d1
    A1[i+2,1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,2+i+2]=-mp.mpf(728)/d1
    A1[i+2,3+i+2]=-mp.mpf(2)/d1
    A1[i+2,4+i+2]=mp.mpf(1)/d1

#------------------------------------------------------------------------------
# time level of first interval
B1[0,0]=A1[0,0]         # first boundary condition
B1[0,1]=A1[0,1]
B1[0,2]=A1[0,2]
B1[0,3]=A1[0,3]
B1[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B1[M+4,i+2]=B1[0,i-(M-2)]

# B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B1[1,1]=A1[1,1]+26*q1
# B1[1,2]=A1[1,2]+66*q1
# B1[1,3]=A1[1,3]+26*q1
# B1[1,4]=A1[1,4]+q1
B1[1,5]=A1[1,5]
B1[1,6]=A1[1,6]
B1[1,7]=A1[1,7]
B1[1,8]=A1[1,8]
B1[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

# B1[2,0]=A1[2,0]+q2         # Tau1
# B1[2,1]=A1[2,1]+237*q2
# B1[2,2]=A1[2,2]+1682*q2
# B1[2,3]=A1[2,3]+1682*q2
# B1[2,4]=A1[2,4]+237*q2
# B1[2,5]=A1[2,5]+q2
B1[2,6]=A1[2,6]
B1[2,7]=A1[2,7]
B1[2,8]=A1[2,8]
B1[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

B1[3,0]=A1[3,0]                                 # y-y"(1)
# B1[3,1]=A1[3,1]+q1
# B1[3,2]=A1[3,2]+26*q1
# B1[3,3]=A1[3,3]+66*q1
# B1[3,4]=A1[3,4]+26*q1
# B1[3,5]=A1[3,5]+q1
B1[3,6]=A1[3,6]
B1[3,7]=A1[3,7]
B1[3,8]=A1[3,8]
B1[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B1[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B1[i+2,3+i+2]=A1[i+2,3+i+2]
    B1[i+2,4+i+2]=A1[i+2,4+i+2]

for n in range(1,N1+1):
    q1=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(120)
    q2=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(3840)

    # B1[0,0]=A1[0,0]         # first boundary condition
    # B1[0,1]=A1[0,1]
    # B1[0,2]=A1[0,2]
    # B1[0,3]=A1[0,3]
    # B1[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B1[M+4,i+2]=B1[0,i-(M-2)]

    B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B1[1,1]=A1[1,1]+26*q1
    B1[1,2]=A1[1,2]+66*q1
    B1[1,3]=A1[1,3]+26*q1
    B1[1,4]=A1[1,4]+q1
    # B1[1,5]=A1[1,5]
    # B1[1,6]=A1[1,6]
    # B1[1,7]=A1[1,7]
    # B1[1,8]=A1[1,8]
    # B1[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

    B1[2,0]=A1[2,0]+q2         # Tau1
    B1[2,1]=A1[2,1]+237*q2
    B1[2,2]=A1[2,2]+1682*q2
    B1[2,3]=A1[2,3]+1682*q2
    B1[2,4]=A1[2,4]+237*q2
    B1[2,5]=A1[2,5]+q2
    # B1[2,6]=A1[2,6]
    # B1[2,7]=A1[2,7]
    # B1[2,8]=A1[2,8]
    # B1[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

    #B1[3,0]=A1[3,0]                                 # y-y"(1)
    B1[3,1]=A1[3,1]+q1
    B1[3,2]=A1[3,2]+26*q1
    B1[3,3]=A1[3,3]+66*q1
    B1[3,4]=A1[3,4]+26*q1
    B1[3,5]=A1[3,5]+q1
    # B1[3,6]=A1[3,6]
    # B1[3,7]=A1[3,7]
    # B1[3,8]=A1[3,8]
    # B1[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

    for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
        # B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B1[i+2,i+2]=A1[i+2,i+2]+66*q1
        B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B1[i+2,3+i+2]=A1[i+2,3+i+2]
        # B1[i+2,4+i+2]=A1[i+2,4+i+2]

    W1[0]=(tF[n]**a_l)*(mp.sin(2*mp.pi*x[0]))
    for i in range(0,M+3):
        s=0
        if n>1:
            s=mp.fdot(u[i,1:n]-u[i,0:n-1],zeta[0:n-1,n]+zeta_s[0:n-1,n])
        W1[i+1]=f(X[i],tF[n])+u[i,n-1]*(zeta[n-1,n]+zeta_s[n-1,n])-s
    W1[M+4]=(tF[n]**a_l)*(mp.sin(2*mp.pi*x[M]))


    # ab = extract_banded_form(B1, lower_bw=8, upper_bw=8)
    # G1 = solve_banded((8, 8), ab, W1)

    # for i in range(0,M+5):
    #     C1[n,i]=G1.item(i)

    B1_mp = mp.matrix(B1)
    W1_mp = mp.matrix(W1)

    G1 = mp.lu_solve(B1_mp, W1_mp)

    for i in range(M+5):
        C1[n, i] = G1[i]

    u[0,n]=(C1[n,0]+C1[n,1]*26+C1[n,2]*66+C1[n,3]*26+C1[n,4])/120
    u[1,n]=(C1[n,0]+C1[n,1]*237+C1[n,2]*1682+C1[n,3]*1682+C1[n,4]*237+C1[n,5])/3840

    for i in range(2,M+1):
        u[i,n]=(C1[n,i-3+2]+C1[n,i-2+2]*26+C1[n,i-1+2]*66+C1[n,i+2]*26+C1[n,i+1+2])/120

    u[M+1,n]=(C1[n,M-3+2]+237*C1[n,M-2+2]+1682*C1[n,M-1+2]+1682*C1[n,M+2]+237*C1[n,M+1+2]+C1[n,M+2+2])/3840
    u[M+2,n]=(C1[n,M-2+2]+C1[n,M-1+2]*26+C1[n,M+2]*66+C1[n,M+1+2]*26+C1[n,M+2+2])/120

#------------------------------------------------------------------------------------------------------
for i in range(0,M+3):
    U[i,1]=u[i,N1]

#Xi=np.zeros((N1+1,N+1))

# Xi = mp.matrix(N1+1, N+1)
# Xi_s = mp.matrix(N1+1, N+1)

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi[k,j]=(((t[j]-tF[k])**(1-a))-((t[j]-tF[k+1])**(1-a)))/((tF[k+1]-tF[k])*g1)

# #Xi_s=np.zeros((N1+1,N+1))

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi_s[k,j]=(((t[j]-tF[k])**(1-a_s))-((t[j]-tF[k+1])**(1-a_s)))/((tF[k+1]-tF[k])*g1_s)

Xi   = mp.matrix(N1 + 1, N + 1)
Xi_s = mp.matrix(N1 + 1, N + 1)

for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi[k, j] = (
            dt1**(1 - a) - dt2**(1 - a)
        ) / (hF * g1)


for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi_s[k, j] = (
            dt1**(1 - a_s) - dt2**(1 - a_s)
        ) / (hF * g1_s)


def gamma_mat(k,j):
    return ((mp.power(t[j] - t[k], 1 - a) - mp.power(t[j] - t[k+1], 1 - a)) / g1)

def gamma_mat_s(k,j):
    return ((mp.power(t[j] - t[k], 1 - a_s) - mp.power(t[j] - t[k+1], 1 - a_s)) / g1_s)

#Del=np.zeros((N+1,N+1))

#for j in range(0,N+1):
    #for k in range(0,j):
        #Del[k,j]=((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

#def Del(k,j):
    #return ((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

def Del(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a))
    term2 = mp.power(t[j] - t[k+1], 1 - a) * (t[j] + (a - 2) * t[k] + (1 - a) * t[k+1])
    return (term1 - term2) / g - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat(k, j)

def Del_s(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a_s))
    term2 = mp.power(t[j] - t[k+1], 1 - a_s) * (t[j] + (a_s - 2) * t[k] + (1 - a_s) * t[k+1])
    return (term1 - term2) / g_s - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat_s(k, j)

def d(k,j):
    if k == j:
        return (
            2 * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

def d_s(k,j):
    if k == j:
        return (
            2 * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat_s(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

B[0,0]=A1[0,0]         # first Boundary condition
B[0,1]=A1[0,1]
B[0,2]=A1[0,2]
B[0,3]=A1[0,3]
B[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B[M+4,i+2]=B[0,i-(M-2)]

# B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B[1,1]=A1[1,1]+26*q1
# B[1,2]=A1[1,2]+66*q1
# B[1,3]=A1[1,3]+26*q1
# B[1,4]=A1[1,4]+q1
B[1,5]=A1[1,5]
B[1,6]=A1[1,6]
B[1,7]=A1[1,7]
B[1,8]=A1[1,8]
B[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B[M+3,i+2]=B[1,9-(i-(-7+M))]

# B[2,0]=A1[2,0]+q2         # Tau1
# B[2,1]=A1[2,1]+237*q2
# B[2,2]=A1[2,2]+1682*q2
# B[2,3]=A1[2,3]+1682*q2
# B[2,4]=A1[2,4]+237*q2
# B[2,5]=A1[2,5]+q2
B[2,6]=A1[2,6]
B[2,7]=A1[2,7]
B[2,8]=A1[2,8]
B[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B[M+2,i+2]=B[2,9-(i-(-7+M))]

B[3,0]=A1[3,0]                                 # y-y"(1)
# B[3,1]=A1[3,1]+q1
# B[3,2]=A1[3,2]+26*q1
# B[3,3]=A1[3,3]+66*q1
# B[3,4]=A1[3,4]+26*q1
# B[3,5]=A1[3,5]+q1
B[3,6]=A1[3,6]
B[3,7]=A1[3,7]
B[3,8]=A1[3,8]
B[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B[i+2,3+i+2]=A1[i+2,3+i+2]
    B[i+2,4+i+2]=A1[i+2,4+i+2]


n=2
# --- D1 matrix ---
D1 = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1[0,0] = -two * mp.mpf(Del(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
D1[1,0] = two * Del(1, j) / (Tau[1] * Tau[2])
D1[2,0] = D1[0,0]+D1[1,0]

# --- D1 matrix ---
D1_s = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1_s[0,0] = -two * mp.mpf(Del_s(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
D1_s[1,0] = two * Del_s(1, j) / (Tau[1] * Tau[2])
D1_s[2,0] = D1_s[0,0]+D1_s[1,0]

q1=((D1[2,0]+D1_s[2,0])-c_)/120
q2=((D1[2,0]+D1_s[2,0])-c_)/3840

# B[0,0]=A1[0,0]         # first Boundary condition
# B[0,1]=A1[0,1]
# B[0,2]=A1[0,2]
# B[0,3]=A1[0,3]
# B[0,4]=A1[0,4]

# for i in range(M-2,M+3):
#     B[M+4,i+2]=B[0,i-(M-2)]

B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
B[1,1]=A1[1,1]+26*q1
B[1,2]=A1[1,2]+66*q1
B[1,3]=A1[1,3]+26*q1
B[1,4]=A1[1,4]+q1
# B[1,5]=A1[1,5]
# B[1,6]=A1[1,6]
# B[1,7]=A1[1,7]
# B[1,8]=A1[1,8]
# B[1,9]=A1[1,9]

for i in range(-7+M,M+3):
    B[M+3,i+2]=B[1,9-(i-(-7+M))]

B[2,0]=A1[2,0]+q2         # Tau1
B[2,1]=A1[2,1]+237*q2
B[2,2]=A1[2,2]+1682*q2
B[2,3]=A1[2,3]+1682*q2
B[2,4]=A1[2,4]+237*q2
B[2,5]=A1[2,5]+q2
# B[2,6]=A1[2,6]
# B[2,7]=A1[2,7]
# B[2,8]=A1[2,8]
# B[2,9]=A1[2,9]

for i in range(-7+M,M+3):
    B[M+2,i+2]=B[2,9-(i-(-7+M))]

# B[3,0]=A1[3,0]                                 # y-y"(1)
B[3,1]=A1[3,1]+q1
B[3,2]=A1[3,2]+26*q1
B[3,3]=A1[3,3]+66*q1
B[3,4]=A1[3,4]+26*q1
B[3,5]=A1[3,5]+q1
# B[3,6]=A1[3,6]
# B[3,7]=A1[3,7]
# B[3,8]=A1[3,8]
# B[3,9]=A1[3,9]

for i in range(-7+M,M+3):
    B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
    # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    B[i+2,i+2]=A1[i+2,i+2]+66*q1
    B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    # B[i+2,3+i+2]=A1[i+2,3+i+2]
    # B[i+2,4+i+2]=A1[i+2,4+i+2]

W[0]=(t[n]**a_l)*(mp.sin(2*mp.pi*x[0]))
for i in range(0,M+3):
    W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot(D1[0:n,0]+D1_s[0:n,0],U[i,0:n])+f(X[i],t[n])
W[M+4]=(t[n]**a_l)*(mp.sin(2*mp.pi*x[M]))

# ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
# G = solve_banded((8, 8), ab, W)

# for i in range(0,M+5):
#     C2[n,i]=G.item(i)

B_mp = mp.matrix(B)
W_mp = mp.matrix(W)

G = mp.lu_solve(B_mp, W_mp)

for i in range(M+5):
     C2[n, i] = G[i]

U[0,n]=(C2[n,0]+C2[n,1]*26+C2[n,2]*66+C2[n,3]*26+C2[n,4])/120
U[1,n]=(C2[n,0]+C2[n,1]*237+C2[n,2]*1682+C2[n,3]*1682+C2[n,4]*237+C2[n,5])/3840

for i in range(2,M+1):
    U[i,n]=(C2[n,i-3+2]+C2[n,i-2+2]*26+C2[n,i-1+2]*66+C2[n,i+2]*26+C2[n,i+1+2])/120

U[M+1,n]=(C2[n,M-3+2]+237*C2[n,M-2+2]+1682*C2[n,M-1+2]+1682*C2[n,M+2]+237*C2[n,M+1+2]+C2[n,M+2+2])/3840
U[M+2,n]=(C2[n,M-2+2]+C2[n,M-1+2]*26+C2[n,M+2]*66+C2[n,M+1+2]*26+C2[n,M+2+2])/120


for n in range(3,N+1):
    # --- D1 matrix ---
    D1 = mp.zeros(N+1, 1)  # ✅ allows mpf/mpc objects
    two=mp.mpf(2)
    j=n
    D1[0,0] = -two * Del(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1[1,0] = (
        -two * Del(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat(2, j) / (Tau[2] + Tau[3])
    )

    if j>3:
        for k in range(2, j - 1):
            D1[k,0] = (
                -two * Del(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
                + two * Del(k, j) / (Tau[k] * Tau[k + 1])
                - two * Del(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
                - gamma_mat(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
                + gamma_mat(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
            )

    D1[j - 1,0] = (
        -two * Del(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1[j,0] = (
        two * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    # --- D1 matrix ---
    D1_s = mp.zeros(N+1,1)
    two=mp.mpf(2)
    j=n
    D1_s[0,0] = -two * Del_s(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1_s[1,0] = (
        -two * Del_s(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del_s(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat_s(2, j) / (Tau[2] + Tau[3])
    )
    for k in range(2, j - 1):
        D1_s[k,0] = (
            -two * Del_s(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
            + two * Del_s(k, j) / (Tau[k] * Tau[k + 1])
            - two * Del_s(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
            - gamma_mat_s(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
            + gamma_mat_s(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
        )
    D1_s[j - 1,0] = (
        -two * Del_s(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del_s(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat_s(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1_s[j,0] = (
        two * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat_s(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    q1=((d(n,n)+d_s(n,n))-c_)/120
    q2=((d(n,n)+d_s(n,n))-c_)/3840

    # B[0,0]=A1[0,0]         # first Boundary condition
    # B[0,1]=A1[0,1]
    # B[0,2]=A1[0,2]
    # B[0,3]=A1[0,3]
    # B[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B[M+4,i+2]=B[0,i-(M-2)]

    B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1
    # B[1,5]=A1[1,5]
    # B[1,6]=A1[1,6]
    # B[1,7]=A1[1,7]
    # B[1,8]=A1[1,8]
    # B[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2
    # B[2,6]=A1[2,6]
    # B[2,7]=A1[2,7]
    # B[2,8]=A1[2,8]
    # B[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    # B[3,0]=A1[3,0]                                 # y-y"(1)
    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1
    # B[3,6]=A1[3,6]
    # B[3,7]=A1[3,7]
    # B[3,8]=A1[3,8]
    # B[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
        # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B[i+2,3+i+2]=A1[i+2,3+i+2]
        # B[i+2,4+i+2]=A1[i+2,4+i+2]

    W[0]=(t[n]**a_l)*(mp.sin(2*mp.pi*x[0]))
    for i in range(0,M+3):
        W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot((D1[0:n,0]+D1_s[0:n,0]),U[i,0:n])+f(X[i],t[n])
    W[M+4]=(t[n]**a_l)*(mp.sin(2*mp.pi*x[M]))

    # ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
    # G = solve_banded((8, 8), ab, W)

    # for i in range(0,M+5):
    #     C[n,i]=G.item(i)

    B_mp = mp.matrix(B)
    W_mp = mp.matrix(W)

    G = mp.lu_solve(B_mp, W_mp)

    for i in range(M+5):
         C[n, i] = G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

    U[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
    U[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

# u_exact=np.zeros((M+3,N1+1))
# u_error=np.zeros((M+3,N1+1))
# for j in range(0,N1+1):
#     for i in range(0,M+3):
#         u_exact[i,j]=(tF[j]**a_l)*(mp.sin(2*mp.pi*X[i]))
#         u_error[i,j]=abs(u_exact[i,j]-u[i,j])

# U_exact=np.zeros((M+3,N+1))
# U_error=np.zeros((M+3,N+1))
U_exact = mp.matrix(M+3, N+1)
U_error = mp.matrix(M+3, N+1)

for i in range(0,M+3):
    for j in range(0,N+1):
        U_exact[i,j]=(t[j]**a_l)*(mp.sin(2*mp.pi*X[i]))
        U_error[i,j]=abs(U_exact[i,j]-U[i,j])

#print(np.max(U_error))
max_error = max(abs(U_error[i,j]) for i in range(U_error.rows) for j in range(U_error.cols))
print(max_error)

# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Meshgrid (coarse grid)
# # --------------------------------------------------
# Xplot_coarse, Yplot_coarse = np.meshgrid(X, t)

# Xplot_coarse = Xplot_coarse.astype(float)
# Yplot_coarse = Yplot_coarse.astype(float)
# V_float1 = np.array(U_exact, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float1.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Exact solution")
# ax.grid(False)

# # Fixed good viewing angle (for paper)
# ax.view_init(elev=28, azim=225)


# # --------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("exact_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("exact_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -------------------------------------------------------------------------------------------------------------------------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Convert numerical solution to float
# # --------------------------------------------------
# V_float = np.array(U, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Numerical solution ")
# ax.grid(False)

# # Fixed good viewing angle (match exact solution plot)
# ax.view_init(elev=28, azim=225)

# # -------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("numerical_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("numerical_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "lines.linewidth": 2,
#     "figure.dpi": 120
# })

# #-------------------------------------------------------------------------------------------------------------------------------------

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# U_error_float = np.array(U_error, dtype=float)

# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, U_error_float.T,
#     cmap='viridis',
#     linewidth=0,
#     antialiased=True,
#     shade=True
# )

# surf.set_rasterized(True)

# ax.set_xlabel(r"Space $x$", labelpad=10)
# ax.set_ylabel(r"Time $t$", labelpad=10)
# ax.set_zlabel(r"Absolute error", labelpad=10)

# ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# ax.ticklabel_format(axis='z', style='sci', scilimits=(0,0))

# ax.view_init(elev=28, azim=225)
# ax.dist = 11
# ax.grid(False)

# plt.tight_layout()
# plt.savefig("Error.png", dpi=300, bbox_inches="tight")
# plt.savefig("Error.pdf", bbox_inches="tight")
# plt.show()

# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)

# end=datetime.now()
# print(end-start)

# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.ticker import ScalarFormatter
# from datetime import datetime
# import mpmath as mp

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120,
# })

# # --------------------------------------------------
# # mpmath -> numpy float conversion
# # --------------------------------------------------
# def to_float(a, vector=False):
#     """Convert mp.matrix / list of mpf / nested lists to a float ndarray."""
#     if isinstance(a, mp.matrix):
#         a = a.tolist()
#     arr = np.array([[float(v) for v in row] for row in a]) \
#         if (len(a) and isinstance(a[0], (list, tuple))) \
#         else np.array([float(v) for v in a])
#     return arr.ravel() if vector else arr

# X_f = to_float(X, vector=True)
# t_f = to_float(t, vector=True)

# Xplot_coarse, Yplot_coarse = np.meshgrid(X_f, t_f)   # shape (len(t), len(X))

# U_exact_f = to_float(U_exact)   # expected shape (len(X), len(t))
# U_f       = to_float(U)
# U_err_f   = to_float(U_error)

# # --------------------------------------------------
# # Generic surface plot
# # --------------------------------------------------
# def surface(Z, zlabel, fname, sci=False, animate=False):
#     fig = plt.figure(figsize=(7.5, 5.5))
#     ax = fig.add_subplot(111, projection="3d")

#     surf = ax.plot_surface(Xplot_coarse, Yplot_coarse, Z.T,
#                            cmap="viridis", linewidth=0,
#                            antialiased=True, shade=True)
#     surf.set_rasterized(True)

#     ax.set_xlabel(r"Space $x$", labelpad=10)
#     ax.set_ylabel(r"Time $t$", labelpad=10)
#     ax.set_zlabel(zlabel, labelpad=10)
#     if sci:
#         ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
#         ax.ticklabel_format(axis="z", style="sci", scilimits=(0, 0))

#     ax.view_init(elev=28, azim=225)
#     ax.set_box_aspect(None, zoom=0.9)   # replaces deprecated ax.dist
#     ax.grid(False)

#     plt.tight_layout()
#     plt.savefig(f"{fname}.png", dpi=300, bbox_inches="tight")
#     plt.savefig(f"{fname}.pdf", bbox_inches="tight")
#     plt.show()

#     if animate:
#         for angle in range(0, 360, 5):
#             ax.view_init(elev=30, azim=angle)
#             plt.draw()
#             plt.pause(0.1)

# surface(U_exact_f, r"Exact solution",     "exact_solution_surface_alpha06")
# surface(U_f,       r"Numerical solution", "numerical_solution_surface_alpha06")
# surface(U_err_f,   r"Absolute error",     "Error", sci=True)

# end = datetime.now()
# print(end - start)















Example 4


In [ ]:
import sys
from operator import mul
from datetime import datetime
from mpmath import mp

start = datetime.now()

# ----------------------------------------------------------------------------
# Settings.  Optional command line:  python multiterm_fast_sin.py M N DPS
# (defaults are your original M=64, N=8000; dps 120 -> 40)
# ----------------------------------------------------------------------------
M   = int(sys.argv[1]) if len(sys.argv) > 1 else 8*2
N   = int(sys.argv[2]) if len(sys.argv) > 2 else 20000
DPS = int(sys.argv[3]) if len(sys.argv) > 3 else 50
mp.dps = DPS
mpf = mp.mpf

a_l = mpf("0.80")
a   = a_l
a_s = mpf("0.40")
g1, g1_s = mp.gamma(2 - a), mp.gamma(2 - a_s)
g,  g_s  = mp.gamma(3 - a), mp.gamma(3 - a_s)
ig1, ig1_s, ig, ig_s = 1/g1, 1/g1_s, 1/g, 1/g_s

a_, c_, L, T = mpf(1), mpf(0), mpf(1), mpf(1)
h = L / M
r = 3 / a - 1
zero = mpf(0)

# ----------------------------------------------------------------------------
# PROBLEM DATA  (exact solution u = tfun(t) * Sfun(x);  f = ftime(t) * Sfun(x))
# ----------------------------------------------------------------------------
def tfun(t):
    return t**a_l

def Sfun(x):
    return mp.sin(2*mp.pi*x)

def ftime(t):
    return (mp.gamma(a_l+1)
            + mp.power(t, a_l-a_s) * (mp.gamma(a_l+1) / mp.gamma(a_l-a_s+1))
            + 4*(mp.pi**2) * (t**a_l))

# ----------------------------------------------------------------------------
# Grids
# ----------------------------------------------------------------------------
t = [T * (mpf(i) / N) ** r for i in range(N + 1)]
N1 = int(mp.ceil(mp.power(mp.power(N, a - 3) / mp.power(t[1], 2 - a), 1 / (a - 2))))
tF = [t[1] * mp.power(mpf(i) / N1, r) for i in range(N1 + 1)]
Tau = [zero] + [t[i] - t[i-1] for i in range(1, N + 1)]

x = [mpf(i) * L / M for i in range(M + 1)]
X = [zero] * (M + 3)
X[0] = x[0]
X[1] = (x[0] + x[1]) / 2
for j in range(2, M + 1):
    X[j] = x[j-1]
X[M+1] = (x[M] + x[M-1]) / 2
X[M+2] = x[M]
S = [Sfun(xi) for xi in X]                 # spatial profile, never changes in time
S_left, S_right = Sfun(x[0]), Sfun(x[M])

# ----------------------------------------------------------------------------
# Matrix pieces (sparse rows, stored as dicts {col: value})
# ----------------------------------------------------------------------------
c1 = mpf(69120) * h**2 / a_
d1 = mpf(4320)  * h**2 / a_

S5 = [1, 26, 66, 26, 1]
S6 = [1, 237, 1682, 1682, 237, 1]
R0 = [mpf(v) / 120 for v in S5]
R1 = [mpf(v) / d1 for v in (-717, -1448, 4300, -1322, -938, 202, -92, 10, 7, -2)]
R2 = [mpf(v) / c1 for v in (-1475, -30149, 31918, 30322, -27776, -3680, 994, -98, -77, 21)]
R3 = [mpf(v) / d1 for v in (2, -725, -1454, 4396, -1574, -602, -50, 4, 4, -1)]
RI = [mpf(v) / d1 for v in (1, -2, -728, -1406, 4270, -1406, -728, -2, 1)]

def mirror(row):
    return {M + 4 - c: v for c, v in row.items()}

def build_rows(q1, q2):
    qa = [q1 * s for s in S5]
    qb = [q2 * s for s in S6]
    rows = [None] * (M + 5)
    rows[0] = dict(enumerate(R0))
    rows[M+4] = mirror(rows[0])
    r1 = dict(enumerate(R1))
    for k in range(5): r1[k] += qa[k]
    r2 = dict(enumerate(R2))
    for k in range(6): r2[k] += qb[k]
    r3 = dict(enumerate(R3))
    for k in range(5): r3[1 + k] += qa[k]
    rows[1], rows[2], rows[3] = r1, r2, r3
    rows[M+3], rows[M+2], rows[M+1] = mirror(r1), mirror(r2), mirror(r3)
    for i in range(4, M + 1):
        row = {i - 4 + k: RI[k] for k in range(9)}
        for k in range(5): row[i - 2 + k] += qa[k]
        rows[i] = row
    return rows

def solve_banded_mp(rows, rhs, kl=8):
    n = len(rows)
    b = list(rhs)
    for k in range(n):
        hi = min(n, k + kl + 1)
        p, best = k, abs(rows[k].get(k, 0))
        for rr in range(k + 1, hi):
            v = abs(rows[rr].get(k, 0))
            if v > best:
                best, p = v, rr
        if p != k:
            rows[k], rows[p] = rows[p], rows[k]
            b[k], b[p] = b[p], b[k]
        rk, piv, bk = rows[k], rows[k][k], b[k]
        for rr in range(k + 1, hi):
            row = rows[rr]
            v = row.pop(k, None)
            if not v:
                continue
            m = v / piv
            for c, val in rk.items():
                if c != k:
                    row[c] = row.get(c, 0) - m * val
            b[rr] -= m * bk
    xs = [zero] * n
    for i in range(n - 1, -1, -1):
        row = rows[i]
        s = b[i]
        for c, v in row.items():
            if c > i:
                s -= v * xs[c]
        xs[i] = s / row[i]
    return xs

def recover(c):
    v = [zero] * (M + 3)
    v[0] = (c[0] + 26*c[1] + 66*c[2] + 26*c[3] + c[4]) / 120
    v[1] = (c[0] + 237*c[1] + 1682*c[2] + 1682*c[3] + 237*c[4] + c[5]) / 3840
    for i in range(2, M + 1):
        v[i] = (c[i-1] + 26*c[i] + 66*c[i+1] + 26*c[i+2] + c[i+3]) / 120
    v[M+1] = (c[M-1] + 237*c[M] + 1682*c[M+1] + 1682*c[M+2] + 237*c[M+3] + c[M+4]) / 3840
    v[M+2] = (c[M] + 26*c[M+1] + 66*c[M+2] + 26*c[M+3] + c[M+4]) / 120
    return v

# ----------------------------------------------------------------------------
# Phase 1: fine graded mesh on [0, t1]
# ----------------------------------------------------------------------------
def zeta_vec(n):
    out = []
    for k in range(n):
        e1, e2 = tF[n] - tF[k], tF[n] - tF[k+1]
        hF = tF[k+1] - tF[k]
        out.append((e1**(1-a)   - e2**(1-a))   / (hF * g1)
                 + (e1**(1-a_s) - e2**(1-a_s)) / (hF * g1_s))
    return out

u = [[zero] for _ in range(M + 3)]
for n in range(1, N1 + 1):
    zc = zeta_vec(n)
    zl = zc[n-1]
    rows = build_rows((zl - c_) / 120, (zl - c_) / 3840)
    tn = tF[n]
    fc = ftime(tn)
    rhs = [zero] * (M + 5)
    rhs[0] = tfun(tn) * S_left
    for i in range(M + 3):
        ui = u[i]
        s = mp.fsum((ui[k+1] - ui[k]) * zc[k] for k in range(n - 1)) if n > 1 else 0
        rhs[i+1] = fc * S[i] + ui[n-1] * zl - s
    rhs[M+4] = tfun(tn) * S_right
    vals = recover(solve_banded_mp(rows, rhs))
    for i in range(M + 3):
        u[i].append(vals[i])

# ----------------------------------------------------------------------------
# Phase 2: coarse graded mesh on [t1, T]
# ----------------------------------------------------------------------------
hFs = [tF[k+1] - tF[k] for k in range(N1)]

def xi_vec(tj):
    out = []
    for k in range(N1):
        e1, e2 = tj - tF[k], tj - tF[k+1]
        out.append((e1**(1-a)   - e2**(1-a))   / (hFs[k] * g1)
                 + (e1**(1-a_s) - e2**(1-a_s)) / (hFs[k] * g1_s))
    return out

# ---- j-independent coefficient tables (computed once) ----------------------
hdT  = [(Tau[k+1] - Tau[k]) / 2 for k in range(N)]
tka  = [(a   - 2) * t[k] + (1 - a)   * t[k+1] for k in range(N)]
tkas = [(a_s - 2) * t[k] + (1 - a_s) * t[k+1] for k in range(N)]
CA = [zero] * (N + 3); CB = [zero] * (N + 3); CC = [zero] * (N + 3); CE = [zero] * (N + 3)
for k in range(1, N + 1):
    CE[k] = 1 / (Tau[k-1] + Tau[k])
for k in range(2, N - 1):
    CA[k] = 2 / (Tau[k] * (Tau[k-1] + Tau[k]))
    CB[k] = 2 / (Tau[k] * Tau[k+1])
    CC[k] = 2 / (Tau[k+1] * (Tau[k+1] + Tau[k+2]))

# If (1-a_s)/(1-a) is an integer m (here 0.6/0.2 = 3), then
# (t_j-t_k)^(1-a_s) = ((t_j-t_k)^(1-a))^m: one mp.power per k instead of two.
m_ratio = (1 - a_s) / (1 - a)
m_int = int(mp.nint(m_ratio))
USE_POWER_TRICK = abs(m_ratio - m_int) < mpf(10)**(-(DPS - 8)) and 1 <= m_int <= 8

def build_D(j):
    """Combined weights D[0..j] = D(alpha) + D_s(alpha_s) for time level j."""
    tj = t[j]
    dt = [tj - t[k] for k in range(j)]
    pw = [mp.power(d_, 1 - a) for d_ in dt] + [zero]
    if USE_POWER_TRICK:
        pws = [p_**m_int for p_ in pw]
    else:
        pws = [mp.power(d_, 1 - a_s) for d_ in dt] + [zero]
    Gm = [zero] * j
    Dl = [zero] * j
    for k in range(j):
        p0, p1 = pw[k], pw[k+1]
        q0, q1_ = pws[k], pws[k+1]
        gm_a  = (p0 - p1) * ig1
        gm_s  = (q0 - q1_) * ig1_s
        Gm[k] = gm_a + gm_s
        Dl[k] = ((p0 * dt[k] - p1 * (tj + tka[k])) * ig
                 + (q0 * dt[k] - q1_ * (tj + tkas[k])) * ig_s
                 - hdT[k] * Gm[k])
    Tn = Tau
    D = [zero] * (j + 1)
    D[0] = -2*Dl[1] / (Tn[1]*(Tn[1]+Tn[2])) + Gm[1] * CE[2]
    if j == 2:      # special first step used in the original code
        D[1] = 2*Dl[1] / (Tn[1]*Tn[2])
        D[2] = D[0] + D[1]
        return D
    D[1] = (-2*Dl[2] / (Tn[2]*(Tn[2]+Tn[3])) + 2*Dl[1] / (Tn[1]*Tn[2])
            + Gm[2] * CE[3])
    for k in range(2, j - 1):
        D[k] = (-CA[k]*Dl[k-1] + CB[k]*Dl[k] - CC[k]*Dl[k+1]
                - CE[k]*Gm[k-1] + CE[k+2]*Gm[k+1])
    D[j-1] = (-2*Dl[j-2] / (Tn[j-1]*(Tn[j-2]+Tn[j-1]))
              + 2*Dl[j-1] / (Tn[j-1]*Tn[j])
              - Gm[j-2] * CE[j-1])
    D[j] = 2*Dl[j-1] / (Tn[j]*(Tn[j-1]+Tn[j])) + Gm[j-1] * CE[j]
    return D

# ---- fixed-point storage of U: the O(N^2 * M) history sums become integer dots
P = int(3.33 * DPS) + 64
def to_fixed(v):
    s, man, ex_, bc = v._mpf_
    sh = ex_ + P
    val = man << sh if sh >= 0 else man >> (-sh)
    return -val if s else val
def from_fixed2(v):                   # value carrying scale 2^(2P)
    return mp.ldexp(mpf(v), -2 * P)

U = [[0, to_fixed(u[i][N1])] for i in range(M + 3)]     # integers, scale 2^P

for n in range(2, N + 1):
    Dsum = build_D(n)
    q1 = (Dsum[n] - c_) / 120
    q2 = (Dsum[n] - c_) / 3840
    rows = build_rows(q1, q2)

    Xc = xi_vec(t[n])
    wh = [zero] * (N1 + 1)
    wh[0] = Xc[0]
    for k in range(1, N1):
        wh[k] = Xc[k] - Xc[k-1]
    wh[N1] = -Xc[N1-1]

    tn = t[n]
    fc = ftime(tn)
    Dn = [to_fixed(d_) for d_ in Dsum[:n]]
    rhs = [zero] * (M + 5)
    rhs[0] = tfun(tn) * S_left
    for i in range(M + 3):
        hist = from_fixed2(sum(map(mul, Dn, U[i])))     # len(U[i]) == n
        rhs[i+1] = mp.fdot(wh, u[i]) + hist + fc * S[i]
    rhs[M+4] = tfun(tn) * S_right

    vals = recover(solve_banded_mp(rows, rhs))
    for i in range(M + 3):
        U[i].append(to_fixed(vals[i]))

# ----------------------------------------------------------------------------
# Error
# ----------------------------------------------------------------------------
tf = [tfun(tj) for tj in t]
max_error = max(abs(tf[j] * S[i] - mp.ldexp(mpf(U[i][j]), -P))
                for i in range(M + 3) for j in range(N + 1))
print("M =", M, " N =", N, " N1 =", N1, " dps =", DPS, " power trick:", USE_POWER_TRICK)
print("max error:", mp.nstr(max_error, 20))
print("time:", datetime.now() - start)



Example_4_new_code

In [ ]:
import sys
from operator import mul
from datetime import datetime
from mpmath import mp

start = datetime.now()

# ----------------------------------------------------------------------------
# Settings.  Optional command line:  python multiterm_fast_sin.py M N DPS
# (defaults are your original M=64, N=8000; dps 120 -> 40)
# ----------------------------------------------------------------------------
M   = int(sys.argv[1]) if len(sys.argv) > 1 else 100
N   = int(sys.argv[2]) if len(sys.argv) > 2 else 8
DPS = int(sys.argv[3]) if len(sys.argv) > 3 else 40
mp.dps = DPS
mpf = mp.mpf

a_l = mpf("0.80")
a   = a_l
a_s = mpf("0.40")
g1, g1_s = mp.gamma(2 - a), mp.gamma(2 - a_s)
g,  g_s  = mp.gamma(3 - a), mp.gamma(3 - a_s)
ig1, ig1_s, ig, ig_s = 1/g1, 1/g1_s, 1/g, 1/g_s

a_, c_, L, T = mpf(1), mpf(0), mpf(1), mpf(1)
h = L / M
r = 3 / a - 1
zero = mpf(0)

# ----------------------------------------------------------------------------
# PROBLEM DATA  (exact solution u = tfun(t) * Sfun(x);  f = ftime(t) * Sfun(x))
# ----------------------------------------------------------------------------
def tfun(t):
    return t**a_l

def Sfun(x):
    return mp.sin(2*mp.pi*x)

def ftime(t):
    return (mp.gamma(a_l+1)
            + mp.power(t, a_l-a_s) * (mp.gamma(a_l+1) / mp.gamma(a_l-a_s+1))
            + 4*(mp.pi**2) * (t**a_l))

# ----------------------------------------------------------------------------
# Grids
# ----------------------------------------------------------------------------
t = [T * (mpf(i) / N) ** r for i in range(N + 1)]
N1 = int(mp.ceil(mp.power(mp.power(N, a - 3) / mp.power(t[1], 2 - a), 1 / (a - 2))))
tF = [t[1] * mp.power(mpf(i) / N1, r) for i in range(N1 + 1)]
Tau = [zero] + [t[i] - t[i-1] for i in range(1, N + 1)]

x = [mpf(i) * L / M for i in range(M + 1)]
X = [zero] * (M + 3)
X[0] = x[0]
X[1] = (x[0] + x[1]) / 2
for j in range(2, M + 1):
    X[j] = x[j-1]
X[M+1] = (x[M] + x[M-1]) / 2
X[M+2] = x[M]
S = [Sfun(xi) for xi in X]                 # spatial profile, never changes in time
S_left, S_right = Sfun(x[0]), Sfun(x[M])

# ----------------------------------------------------------------------------
# Matrix pieces (sparse rows, stored as dicts {col: value})
# ----------------------------------------------------------------------------
c1 = mpf(69120) * h**2 / a_
d1 = mpf(4320)  * h**2 / a_

S5 = [1, 26, 66, 26, 1]
S6 = [1, 237, 1682, 1682, 237, 1]
R0 = [mpf(v) / 120 for v in S5]
R1 = [mpf(v) / d1 for v in (-717, -1448, 4300, -1322, -938, 202, -92, 10, 7, -2)]
R2 = [mpf(v) / c1 for v in (-1475, -30149, 31918, 30322, -27776, -3680, 994, -98, -77, 21)]
R3 = [mpf(v) / d1 for v in (2, -725, -1454, 4396, -1574, -602, -50, 4, 4, -1)]
RI = [mpf(v) / d1 for v in (1, -2, -728, -1406, 4270, -1406, -728, -2, 1)]

def mirror(row):
    return {M + 4 - c: v for c, v in row.items()}

def build_rows(q1, q2):
    qa = [q1 * s for s in S5]
    qb = [q2 * s for s in S6]
    rows = [None] * (M + 5)
    rows[0] = dict(enumerate(R0))
    rows[M+4] = mirror(rows[0])
    r1 = dict(enumerate(R1))
    for k in range(5): r1[k] += qa[k]
    r2 = dict(enumerate(R2))
    for k in range(6): r2[k] += qb[k]
    r3 = dict(enumerate(R3))
    for k in range(5): r3[1 + k] += qa[k]
    rows[1], rows[2], rows[3] = r1, r2, r3
    rows[M+3], rows[M+2], rows[M+1] = mirror(r1), mirror(r2), mirror(r3)
    for i in range(4, M + 1):
        row = {i - 4 + k: RI[k] for k in range(9)}
        for k in range(5): row[i - 2 + k] += qa[k]
        rows[i] = row
    return rows

def solve_banded_mp(rows, rhs, kl=8):
    n = len(rows)
    b = list(rhs)
    for k in range(n):
        hi = min(n, k + kl + 1)
        p, best = k, abs(rows[k].get(k, 0))
        for rr in range(k + 1, hi):
            v = abs(rows[rr].get(k, 0))
            if v > best:
                best, p = v, rr
        if p != k:
            rows[k], rows[p] = rows[p], rows[k]
            b[k], b[p] = b[p], b[k]
        rk, piv, bk = rows[k], rows[k][k], b[k]
        for rr in range(k + 1, hi):
            row = rows[rr]
            v = row.pop(k, None)
            if not v:
                continue
            m = v / piv
            for c, val in rk.items():
                if c != k:
                    row[c] = row.get(c, 0) - m * val
            b[rr] -= m * bk
    xs = [zero] * n
    for i in range(n - 1, -1, -1):
        row = rows[i]
        s = b[i]
        for c, v in row.items():
            if c > i:
                s -= v * xs[c]
        xs[i] = s / row[i]
    return xs

def recover(c):
    v = [zero] * (M + 3)
    v[0] = (c[0] + 26*c[1] + 66*c[2] + 26*c[3] + c[4]) / 120
    v[1] = (c[0] + 237*c[1] + 1682*c[2] + 1682*c[3] + 237*c[4] + c[5]) / 3840
    for i in range(2, M + 1):
        v[i] = (c[i-1] + 26*c[i] + 66*c[i+1] + 26*c[i+2] + c[i+3]) / 120
    v[M+1] = (c[M-1] + 237*c[M] + 1682*c[M+1] + 1682*c[M+2] + 237*c[M+3] + c[M+4]) / 3840
    v[M+2] = (c[M] + 26*c[M+1] + 66*c[M+2] + 26*c[M+3] + c[M+4]) / 120
    return v

# ----------------------------------------------------------------------------
# Phase 1: fine graded mesh on [0, t1]
# ----------------------------------------------------------------------------
def zeta_vec(n):
    out = []
    for k in range(n):
        e1, e2 = tF[n] - tF[k], tF[n] - tF[k+1]
        hF = tF[k+1] - tF[k]
        out.append((e1**(1-a)   - e2**(1-a))   / (hF * g1)
                 + (e1**(1-a_s) - e2**(1-a_s)) / (hF * g1_s))
    return out

u = [[zero] for _ in range(M + 3)]
for n in range(1, N1 + 1):
    zc = zeta_vec(n)
    zl = zc[n-1]
    rows = build_rows((zl - c_) / 120, (zl - c_) / 3840)
    tn = tF[n]
    fc = ftime(tn)
    rhs = [zero] * (M + 5)
    rhs[0] = tfun(tn) * S_left
    for i in range(M + 3):
        ui = u[i]
        s = mp.fsum((ui[k+1] - ui[k]) * zc[k] for k in range(n - 1)) if n > 1 else 0
        rhs[i+1] = fc * S[i] + ui[n-1] * zl - s
    rhs[M+4] = tfun(tn) * S_right
    vals = recover(solve_banded_mp(rows, rhs))
    for i in range(M + 3):
        u[i].append(vals[i])

# ----------------------------------------------------------------------------
# Phase 2: coarse graded mesh on [t1, T]
# ----------------------------------------------------------------------------
hFs = [tF[k+1] - tF[k] for k in range(N1)]

def xi_vec(tj):
    out = []
    for k in range(N1):
        e1, e2 = tj - tF[k], tj - tF[k+1]
        out.append((e1**(1-a)   - e2**(1-a))   / (hFs[k] * g1)
                 + (e1**(1-a_s) - e2**(1-a_s)) / (hFs[k] * g1_s))
    return out

# ---- j-independent coefficient tables (computed once) ----------------------
hdT  = [(Tau[k+1] - Tau[k]) / 2 for k in range(N)]
tka  = [(a   - 2) * t[k] + (1 - a)   * t[k+1] for k in range(N)]
tkas = [(a_s - 2) * t[k] + (1 - a_s) * t[k+1] for k in range(N)]
CA = [zero] * (N + 3); CB = [zero] * (N + 3); CC = [zero] * (N + 3); CE = [zero] * (N + 3)
for k in range(1, N + 1):
    CE[k] = 1 / (Tau[k-1] + Tau[k])
for k in range(2, N - 1):
    CA[k] = 2 / (Tau[k] * (Tau[k-1] + Tau[k]))
    CB[k] = 2 / (Tau[k] * Tau[k+1])
    CC[k] = 2 / (Tau[k+1] * (Tau[k+1] + Tau[k+2]))

# If (1-a_s)/(1-a) is an integer m (here 0.6/0.2 = 3), then
# (t_j-t_k)^(1-a_s) = ((t_j-t_k)^(1-a))^m: one mp.power per k instead of two.
m_ratio = (1 - a_s) / (1 - a)
m_int = int(mp.nint(m_ratio))
USE_POWER_TRICK = abs(m_ratio - m_int) < mpf(10)**(-(DPS - 8)) and 1 <= m_int <= 8

def build_D(j):
    """Combined weights D[0..j] = D(alpha) + D_s(alpha_s) for time level j."""
    tj = t[j]
    dt = [tj - t[k] for k in range(j)]
    pw = [mp.power(d_, 1 - a) for d_ in dt] + [zero]
    if USE_POWER_TRICK:
        pws = [p_**m_int for p_ in pw]
    else:
        pws = [mp.power(d_, 1 - a_s) for d_ in dt] + [zero]
    Gm = [zero] * j
    Dl = [zero] * j
    for k in range(j):
        p0, p1 = pw[k], pw[k+1]
        q0, q1_ = pws[k], pws[k+1]
        gm_a  = (p0 - p1) * ig1
        gm_s  = (q0 - q1_) * ig1_s
        Gm[k] = gm_a + gm_s
        Dl[k] = ((p0 * dt[k] - p1 * (tj + tka[k])) * ig
                 + (q0 * dt[k] - q1_ * (tj + tkas[k])) * ig_s
                 - hdT[k] * Gm[k])
    Tn = Tau
    D = [zero] * (j + 1)
    D[0] = -2*Dl[1] / (Tn[1]*(Tn[1]+Tn[2])) + Gm[1] * CE[2]
    if j == 2:      # special first step used in the original code
        D[1] = 2*Dl[1] / (Tn[1]*Tn[2])
        D[2] = D[0] + D[1]
        return D
    D[1] = (-2*Dl[2] / (Tn[2]*(Tn[2]+Tn[3])) + 2*Dl[1] / (Tn[1]*Tn[2])
            + Gm[2] * CE[3])
    for k in range(2, j - 1):
        D[k] = (-CA[k]*Dl[k-1] + CB[k]*Dl[k] - CC[k]*Dl[k+1]
                - CE[k]*Gm[k-1] + CE[k+2]*Gm[k+1])
    D[j-1] = (-2*Dl[j-2] / (Tn[j-1]*(Tn[j-2]+Tn[j-1]))
              + 2*Dl[j-1] / (Tn[j-1]*Tn[j])
              - Gm[j-2] * CE[j-1])
    D[j] = 2*Dl[j-1] / (Tn[j]*(Tn[j-1]+Tn[j])) + Gm[j-1] * CE[j]
    return D

# ---- fixed-point storage of U: the O(N^2 * M) history sums become integer dots
P = int(3.33 * DPS) + 64
def to_fixed(v):
    s, man, ex_, bc = v._mpf_
    sh = ex_ + P
    val = man << sh if sh >= 0 else man >> (-sh)
    return -val if s else val
def from_fixed2(v):                   # value carrying scale 2^(2P)
    return mp.ldexp(mpf(v), -2 * P)

U = [[0, to_fixed(u[i][N1])] for i in range(M + 3)]     # integers, scale 2^P

for n in range(2, N + 1):
    Dsum = build_D(n)
    q1 = (Dsum[n] - c_) / 120
    q2 = (Dsum[n] - c_) / 3840
    rows = build_rows(q1, q2)

    Xc = xi_vec(t[n])
    wh = [zero] * (N1 + 1)
    wh[0] = Xc[0]
    for k in range(1, N1):
        wh[k] = Xc[k] - Xc[k-1]
    wh[N1] = -Xc[N1-1]

    tn = t[n]
    fc = ftime(tn)
    Dn = [to_fixed(d_) for d_ in Dsum[:n]]
    rhs = [zero] * (M + 5)
    rhs[0] = tfun(tn) * S_left
    for i in range(M + 3):
        hist = from_fixed2(sum(map(mul, Dn, U[i])))     # len(U[i]) == n
        rhs[i+1] = mp.fdot(wh, u[i]) + hist + fc * S[i]
    rhs[M+4] = tfun(tn) * S_right

    vals = recover(solve_banded_mp(rows, rhs))
    for i in range(M + 3):
        U[i].append(to_fixed(vals[i]))

# ----------------------------------------------------------------------------
# Error
# ----------------------------------------------------------------------------
tf = [tfun(tj) for tj in t]
max_error = max(abs(tf[j] * S[i] - mp.ldexp(mpf(U[i][j]), -P))
                for i in range(M + 3) for j in range(N + 1))
print("M =", M, " N =", N, " N1 =", N1, " dps =", DPS, " power trick:", USE_POWER_TRICK)
print("max error:", mp.nstr(max_error, 20))
print("time:", datetime.now() - start)



Example_4_new_code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
#from scipy.linalg import solve
from datetime import datetime
start = datetime.now()
from scipy.linalg import solve_banded
from mpmath import mp
#from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import ScalarFormatter

mp.dps = 120;

a = mp.mpf("0.80")
a_s = mp.mpf("0.40")

g1 = mp.gamma(2 - a)
g1_s = mp.gamma(2 - a_s)
g = mp.gamma(3 - a)
g_s = mp.gamma(3 - a_s)

def f(x, t):
    x = mp.mpf(x)
    t = mp.mpf(t)
    return (((2*mp.power(t,2-a))/g)+((2*mp.power(t,2-a_s))/g_s)-t**2)*mp.exp(x)

a_ = mp.mpf('1')
c_ = mp.mpf('0')
L  = mp.mpf('1')
T  = mp.mpf('1')

M=8*2*2*2*2
#N=math.floor(T*((M/L)**(6/(3-(a+a_s)/2))))
N=100*2*2*2*2
h = mp.mpf(L) / mp.mpf(M)
r=mp.mpf(3/a)-1
#r=mp.mpf(1)

#t=[T*((i/N)**r) for i in range(0,N+1)]
#t = T * (np.linspace(0, 1, N+1) ** r)
t = [T * (mp.mpf(i) / N) ** r for i in range(N + 1)]

#N1=math.ceil((mp.power(N,a-3)/t[1]**(2-a))**(1/(a-2)))
N1 = int(mp.ceil(
    mp.power(
        mp.power(N, a - 3) / mp.power(t[1], 2 - a),
        1 / (a - 2)
    )
))

#tF=[t[1]*((i/N1)**r) for i in range(0,N1+1)]
#tF = t[1] * np.linspace(0, 1, N1+1) ** r
tF = [t[1] * mp.power(mp.mpf(i) / N1, r) for i in range(N1 + 1)]

# Tau=np.zeros(N+1)
# for i in range(1,N+1):
#     Tau[i]=t[i]-t[i-1]
#Tau = np.diff(t, prepend=0)  # prepend=0  t[−1] = 0
Tau = [mp.mpf('0')] + [t[i] - t[i-1] for i in range(1, len(t))]

#x=[i*h for i in range(0,M+1)]
# x = np.linspace(0, L, M+1)
# X=np.zeros(M+3)

x = [mp.mpf(i) * L / M for i in range(M+1)]
X = [mp.mpf('0') for _ in range(M+3)]

X[0]=x[0]
X[1]=(x[0]+x[1])/2
for j in range(2,M+1):
    X[j]=x[j-1]
X[M+1]=(x[M]+x[M-1])/2
X[M+2]=x[M]

# u=np.zeros((M+3,N1+1))
# U=np.zeros((M+3,N+1))
# exact=np.zeros((M+3,N+1))
# error=np.zeros((M+3,N+1))

# A1 = np.zeros((M+5, M+5))
# B1=np.zeros((M+5,M+5))
# B=np.zeros((M+5,M+5))
# C=np.zeros((N+1,M+5))
# C1=np.zeros((N+1,M+5))
# C2=np.zeros((N+1,M+5))
# W1=np.zeros(M+5)
# W=np.zeros(M+5)

u     = mp.matrix(M+3, N1+1)
U     = mp.matrix(M+3, N+1)
exact = mp.matrix(M+3, N+1)
error = mp.matrix(M+3, N+1)

A1 = mp.matrix(M+5, M+5)
B1 = mp.matrix(M+5, M+5)
B  = mp.matrix(M+5, M+5)

C  = mp.matrix(N+1, M+5)
C1 = mp.matrix(N+1, M+5)
C2 = mp.matrix(N+1, M+5)

W1 = mp.matrix(M+5, 1)
W  = mp.matrix(M+5, 1)

zeta   = mp.matrix(N1+1, N1+1)
zeta_s = mp.matrix(N1+1, N1+1)

c1 = mp.mpf(69120) * h**2 / mp.mpf(a_)
d1 = mp.mpf(4320) * h**2 / mp.mpf(a_)

# --- Compute zeta ---
# for n in range(1, N1 + 1):
#     dt = tF[n] - tF[:n]
#     dt1 = tF[n] - tF[1:n + 1]
#     zeta[:n, n] = (dt**(1 - a) - dt1**(1 - a)) / ((tF[1:n + 1] - tF[:n]) * g1)

for n in range(0,N1+1):
    for k in range(0,n):
        zeta[k,n] = ((mp.power(tF[n] - tF[k], 1-a) - mp.power(tF[n] - tF[k+1], 1-a)) /
                     ((tF[k+1] - tF[k]) * g1))

for n in range(0,N1+1):
    for k in range(0,n):
        zeta_s[k,n] = ((mp.power(tF[n] - tF[k], 1-a_s) - mp.power(tF[n] - tF[k+1], 1-a_s)) /
                     ((tF[k+1] - tF[k]) * g1_s))

for i in range(0,M+3):
    u[i,0]= 0
    U[i,0]=0

A1[0,0] = mp.mpf(1)/120   # first boundary condition
A1[0,1] = mp.mpf(26)/120
A1[0,2] = mp.mpf(66)/120
A1[0,3] = mp.mpf(26)/120
A1[0,4] = mp.mpf(1)/120

for i in range(M-2,M+3):
    A1[M+4,i+2]=A1[0,i-(M-2)]

A1[1,0]=-mp.mpf(717)/d1      # Equation with node x_0  # y-y"(0)
A1[1,1]=-mp.mpf(1448)/d1
A1[1,2]=mp.mpf(4300)/d1
A1[1,3]=-mp.mpf(1322)/d1
A1[1,4]=-mp.mpf(938)/d1
A1[1,5]=mp.mpf(202)/d1
A1[1,6]=-mp.mpf(92)/d1
A1[1,7]=mp.mpf(10)/d1
A1[1,8]=mp.mpf(7)/d1
A1[1,9]=-mp.mpf(2)/d1

for i in range(-7+M,M+3):
    A1[M+3,i+2]=A1[1,9-(i-(-7+M))]

A1[2,0]=-mp.mpf(1475)/c1         # Tau1
A1[2,1]=-mp.mpf(30149)/c1
A1[2,2]=mp.mpf(31918)/c1
A1[2,3]=mp.mpf(30322)/c1
A1[2,4]=-mp.mpf(27776)/c1
A1[2,5]=-mp.mpf(3680)/c1
A1[2,6]=mp.mpf(994)/c1
A1[2,7]=-mp.mpf(98)/c1
A1[2,8]=-mp.mpf(77)/c1
A1[2,9]=mp.mpf(21)/c1

for i in range(-7+M,M+3):
    A1[M+2,i+2]=A1[2,9-(i-(-7+M))]

A1[3,0]=mp.mpf(2)/d1                                # y-y"(1)
A1[3,1]=-mp.mpf(725)/d1
A1[3,2]=-mp.mpf(1454)/d1
A1[3,3]=mp.mpf(4396)/d1
A1[3,4]=-mp.mpf(1574)/d1
A1[3,5]=-mp.mpf(602)/d1
A1[3,6]=-mp.mpf(50)/d1
A1[3,7]=mp.mpf(4)/d1
A1[3,8]=mp.mpf(4)/d1
A1[3,9]=-mp.mpf(1)/d1

for i in range(-7+M,M+3):
    A1[M+1,i+2]=A1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    A1[i+2,-4+i+2]=mp.mpf(1)/d1
    A1[i+2,-3+i+2]=-mp.mpf(2)/d1
    A1[i+2,-2+i+2]=-mp.mpf(728)/d1
    A1[i+2,-1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,i+2]=mp.mpf(4270)/d1
    A1[i+2,1+i+2]=-mp.mpf(1406)/d1
    A1[i+2,2+i+2]=-mp.mpf(728)/d1
    A1[i+2,3+i+2]=-mp.mpf(2)/d1
    A1[i+2,4+i+2]=mp.mpf(1)/d1

#------------------------------------------------------------------------------
# time level of first interval
B1[0,0]=A1[0,0]         # first boundary condition
B1[0,1]=A1[0,1]
B1[0,2]=A1[0,2]
B1[0,3]=A1[0,3]
B1[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B1[M+4,i+2]=B1[0,i-(M-2)]

# B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B1[1,1]=A1[1,1]+26*q1
# B1[1,2]=A1[1,2]+66*q1
# B1[1,3]=A1[1,3]+26*q1
# B1[1,4]=A1[1,4]+q1
B1[1,5]=A1[1,5]
B1[1,6]=A1[1,6]
B1[1,7]=A1[1,7]
B1[1,8]=A1[1,8]
B1[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

# B1[2,0]=A1[2,0]+q2         # Tau1
# B1[2,1]=A1[2,1]+237*q2
# B1[2,2]=A1[2,2]+1682*q2
# B1[2,3]=A1[2,3]+1682*q2
# B1[2,4]=A1[2,4]+237*q2
# B1[2,5]=A1[2,5]+q2
B1[2,6]=A1[2,6]
B1[2,7]=A1[2,7]
B1[2,8]=A1[2,8]
B1[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

B1[3,0]=A1[3,0]                                 # y-y"(1)
# B1[3,1]=A1[3,1]+q1
# B1[3,2]=A1[3,2]+26*q1
# B1[3,3]=A1[3,3]+66*q1
# B1[3,4]=A1[3,4]+26*q1
# B1[3,5]=A1[3,5]+q1
B1[3,6]=A1[3,6]
B1[3,7]=A1[3,7]
B1[3,8]=A1[3,8]
B1[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B1[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B1[i+2,3+i+2]=A1[i+2,3+i+2]
    B1[i+2,4+i+2]=A1[i+2,4+i+2]

for n in range(1,N1+1):
    q1=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(120)
    q2=((zeta[n-1,n]+zeta_s[n-1,n])-c_)/mp.mpf(3840)

    # B1[0,0]=A1[0,0]         # first boundary condition
    # B1[0,1]=A1[0,1]
    # B1[0,2]=A1[0,2]
    # B1[0,3]=A1[0,3]
    # B1[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B1[M+4,i+2]=B1[0,i-(M-2)]

    B1[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B1[1,1]=A1[1,1]+26*q1
    B1[1,2]=A1[1,2]+66*q1
    B1[1,3]=A1[1,3]+26*q1
    B1[1,4]=A1[1,4]+q1
    # B1[1,5]=A1[1,5]
    # B1[1,6]=A1[1,6]
    # B1[1,7]=A1[1,7]
    # B1[1,8]=A1[1,8]
    # B1[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B1[M+3,i+2]=B1[1,9-(i-(-7+M))]

    B1[2,0]=A1[2,0]+q2         # Tau1
    B1[2,1]=A1[2,1]+237*q2
    B1[2,2]=A1[2,2]+1682*q2
    B1[2,3]=A1[2,3]+1682*q2
    B1[2,4]=A1[2,4]+237*q2
    B1[2,5]=A1[2,5]+q2
    # B1[2,6]=A1[2,6]
    # B1[2,7]=A1[2,7]
    # B1[2,8]=A1[2,8]
    # B1[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B1[M+2,i+2]=B1[2,9-(i-(-7+M))]

    #B1[3,0]=A1[3,0]                                 # y-y"(1)
    B1[3,1]=A1[3,1]+q1
    B1[3,2]=A1[3,2]+26*q1
    B1[3,3]=A1[3,3]+66*q1
    B1[3,4]=A1[3,4]+26*q1
    B1[3,5]=A1[3,5]+q1
    # B1[3,6]=A1[3,6]
    # B1[3,7]=A1[3,7]
    # B1[3,8]=A1[3,8]
    # B1[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B1[M+1,i+2]=B1[3,9-(i-(-7+M))]

    for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
        # B1[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B1[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B1[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B1[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B1[i+2,i+2]=A1[i+2,i+2]+66*q1
        B1[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B1[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B1[i+2,3+i+2]=A1[i+2,3+i+2]
        # B1[i+2,4+i+2]=A1[i+2,4+i+2]

    W1[0]=tF[n]**2
    for i in range(0,M+3):
        s=0
        if n>1:
            s=mp.fdot(u[i,1:n]-u[i,0:n-1],zeta[0:n-1,n]+zeta_s[0:n-1,n])
        W1[i+1]=f(X[i],tF[n])+u[i,n-1]*(zeta[n-1,n]+zeta_s[n-1,n])-s
    W1[M+4]=(tF[n]**2)*mp.exp(1)


    # ab = extract_banded_form(B1, lower_bw=8, upper_bw=8)
    # G1 = solve_banded((8, 8), ab, W1)

    # for i in range(0,M+5):
    #     C1[n,i]=G1.item(i)

    B1_mp = mp.matrix(B1)
    W1_mp = mp.matrix(W1)

    G1 = mp.lu_solve(B1_mp, W1_mp)

    for i in range(M+5):
        C1[n, i] = G1[i]

    u[0,n]=(C1[n,0]+C1[n,1]*26+C1[n,2]*66+C1[n,3]*26+C1[n,4])/120
    u[1,n]=(C1[n,0]+C1[n,1]*237+C1[n,2]*1682+C1[n,3]*1682+C1[n,4]*237+C1[n,5])/3840

    for i in range(2,M+1):
        u[i,n]=(C1[n,i-3+2]+C1[n,i-2+2]*26+C1[n,i-1+2]*66+C1[n,i+2]*26+C1[n,i+1+2])/120

    u[M+1,n]=(C1[n,M-3+2]+237*C1[n,M-2+2]+1682*C1[n,M-1+2]+1682*C1[n,M+2]+237*C1[n,M+1+2]+C1[n,M+2+2])/3840
    u[M+2,n]=(C1[n,M-2+2]+C1[n,M-1+2]*26+C1[n,M+2]*66+C1[n,M+1+2]*26+C1[n,M+2+2])/120

#------------------------------------------------------------------------------------------------------
for i in range(0,M+3):
    U[i,1]=u[i,N1]

#Xi=np.zeros((N1+1,N+1))

# Xi = mp.matrix(N1+1, N+1)
# Xi_s = mp.matrix(N1+1, N+1)

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi[k,j]=(((t[j]-tF[k])**(1-a))-((t[j]-tF[k+1])**(1-a)))/((tF[k+1]-tF[k])*g1)

# #Xi_s=np.zeros((N1+1,N+1))

# for j in range(2,N+1):
#     for k in range(0,N1):
#         Xi_s[k,j]=(((t[j]-tF[k])**(1-a_s))-((t[j]-tF[k+1])**(1-a_s)))/((tF[k+1]-tF[k])*g1_s)

Xi   = mp.matrix(N1 + 1, N + 1)
Xi_s = mp.matrix(N1 + 1, N + 1)

for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi[k, j] = (
            dt1**(1 - a) - dt2**(1 - a)
        ) / (hF * g1)


for j in range(2, N + 1):
    tj = t[j]

    for k in range(N1):
        dt1 = tj - tF[k]
        dt2 = tj - tF[k + 1]
        hF  = tF[k + 1] - tF[k]

        Xi_s[k, j] = (
            dt1**(1 - a_s) - dt2**(1 - a_s)
        ) / (hF * g1_s)


def gamma_mat(k,j):
    return ((mp.power(t[j] - t[k], 1 - a) - mp.power(t[j] - t[k+1], 1 - a)) / g1)

def gamma_mat_s(k,j):
    return ((mp.power(t[j] - t[k], 1 - a_s) - mp.power(t[j] - t[k+1], 1 - a_s)) / g1_s)

#Del=np.zeros((N+1,N+1))

#for j in range(0,N+1):
    #for k in range(0,j):
        #Del[k,j]=((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

#def Del(k,j):
    #return ((((t[j]-t[k])**(2-a))-((t[j]-t[k+1])**(1-a))*(t[j]+(a-2)*t[k]+(1-a)*t[k+1]))/g)-((Tau[k+1]-Tau[k])/2)*gamma_mat(k,j)

def Del(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a))
    term2 = mp.power(t[j] - t[k+1], 1 - a) * (t[j] + (a - 2) * t[k] + (1 - a) * t[k+1])
    return (term1 - term2) / g - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat(k, j)

def Del_s(k,j):
    term1 = (mp.power(t[j] - t[k], 2 - a_s))
    term2 = mp.power(t[j] - t[k+1], 1 - a_s) * (t[j] + (a_s - 2) * t[k] + (1 - a_s) * t[k+1])
    return (term1 - term2) / g_s - ((Tau[k+1] - Tau[k]) / 2) * gamma_mat_s(k, j)

def d(k,j):
    if k == j:
        return (
            2 * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

def d_s(k,j):
    if k == j:
        return (
            2 * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
            + gamma_mat_s(j - 1, j) / (Tau[j - 1] + Tau[j])
        )
    else:
        return 0  # d[k,j] = 0 when k > j or k < 0

B[0,0]=A1[0,0]         # first Boundary condition
B[0,1]=A1[0,1]
B[0,2]=A1[0,2]
B[0,3]=A1[0,3]
B[0,4]=A1[0,4]

for i in range(M-2,M+3):
    B[M+4,i+2]=B[0,i-(M-2)]

# B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
# B[1,1]=A1[1,1]+26*q1
# B[1,2]=A1[1,2]+66*q1
# B[1,3]=A1[1,3]+26*q1
# B[1,4]=A1[1,4]+q1
B[1,5]=A1[1,5]
B[1,6]=A1[1,6]
B[1,7]=A1[1,7]
B[1,8]=A1[1,8]
B[1,9]=A1[1,9]

# for i in range(-7+M,M+3):
#     B[M+3,i+2]=B[1,9-(i-(-7+M))]

# B[2,0]=A1[2,0]+q2         # Tau1
# B[2,1]=A1[2,1]+237*q2
# B[2,2]=A1[2,2]+1682*q2
# B[2,3]=A1[2,3]+1682*q2
# B[2,4]=A1[2,4]+237*q2
# B[2,5]=A1[2,5]+q2
B[2,6]=A1[2,6]
B[2,7]=A1[2,7]
B[2,8]=A1[2,8]
B[2,9]=A1[2,9]

# for i in range(-7+M,M+3):
#     B[M+2,i+2]=B[2,9-(i-(-7+M))]

B[3,0]=A1[3,0]                                 # y-y"(1)
# B[3,1]=A1[3,1]+q1
# B[3,2]=A1[3,2]+26*q1
# B[3,3]=A1[3,3]+66*q1
# B[3,4]=A1[3,4]+26*q1
# B[3,5]=A1[3,5]+q1
B[3,6]=A1[3,6]
B[3,7]=A1[3,7]
B[3,8]=A1[3,8]
B[3,9]=A1[3,9]

# for i in range(-7+M,M+3):
#     B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                          # y-y"(i)      i from 2 to M-2
    B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    # B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    # B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    # B[i+2,i+2]=A1[i+2,i+2]+66*q1
    # B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    # B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    B[i+2,3+i+2]=A1[i+2,3+i+2]
    B[i+2,4+i+2]=A1[i+2,4+i+2]


n=2
# --- D1 matrix ---
D1 = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1[0,0] = -two * mp.mpf(Del(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
D1[1,0] = two * Del(1, j) / (Tau[1] * Tau[2])
D1[2,0] = D1[0,0]+D1[1,0]

# --- D1 matrix ---
D1_s = mp.zeros(N+1,1)
two=mp.mpf(2)
j=n
D1_s[0,0] = -two * mp.mpf(Del_s(1, j)) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
D1_s[1,0] = two * Del_s(1, j) / (Tau[1] * Tau[2])
D1_s[2,0] = D1_s[0,0]+D1_s[1,0]

q1=((D1[2,0]+D1_s[2,0])-c_)/120
q2=((D1[2,0]+D1_s[2,0])-c_)/3840

# B[0,0]=A1[0,0]         # first Boundary condition
# B[0,1]=A1[0,1]
# B[0,2]=A1[0,2]
# B[0,3]=A1[0,3]
# B[0,4]=A1[0,4]

# for i in range(M-2,M+3):
#     B[M+4,i+2]=B[0,i-(M-2)]

B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
B[1,1]=A1[1,1]+26*q1
B[1,2]=A1[1,2]+66*q1
B[1,3]=A1[1,3]+26*q1
B[1,4]=A1[1,4]+q1
# B[1,5]=A1[1,5]
# B[1,6]=A1[1,6]
# B[1,7]=A1[1,7]
# B[1,8]=A1[1,8]
# B[1,9]=A1[1,9]

for i in range(-7+M,M+3):
    B[M+3,i+2]=B[1,9-(i-(-7+M))]

B[2,0]=A1[2,0]+q2         # Tau1
B[2,1]=A1[2,1]+237*q2
B[2,2]=A1[2,2]+1682*q2
B[2,3]=A1[2,3]+1682*q2
B[2,4]=A1[2,4]+237*q2
B[2,5]=A1[2,5]+q2
# B[2,6]=A1[2,6]
# B[2,7]=A1[2,7]
# B[2,8]=A1[2,8]
# B[2,9]=A1[2,9]

for i in range(-7+M,M+3):
    B[M+2,i+2]=B[2,9-(i-(-7+M))]

# B[3,0]=A1[3,0]                                 # y-y"(1)
B[3,1]=A1[3,1]+q1
B[3,2]=A1[3,2]+26*q1
B[3,3]=A1[3,3]+66*q1
B[3,4]=A1[3,4]+26*q1
B[3,5]=A1[3,5]+q1
# B[3,6]=A1[3,6]
# B[3,7]=A1[3,7]
# B[3,8]=A1[3,8]
# B[3,9]=A1[3,9]

for i in range(-7+M,M+3):
    B[M+1,i+2]=B[3,9-(i-(-7+M))]

for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
    # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
    # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
    B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
    B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
    B[i+2,i+2]=A1[i+2,i+2]+66*q1
    B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
    B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
    # B[i+2,3+i+2]=A1[i+2,3+i+2]
    # B[i+2,4+i+2]=A1[i+2,4+i+2]

W[0]=t[n]**2
for i in range(0,M+3):
    W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot(D1[0:n,0]+D1_s[0:n,0],U[i,0:n])+f(X[i],t[n])
W[M+4]=(t[n]**2)*mp.exp(1)

# ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
# G = solve_banded((8, 8), ab, W)

# for i in range(0,M+5):
#     C2[n,i]=G.item(i)

B_mp = mp.matrix(B)
W_mp = mp.matrix(W)

G = mp.lu_solve(B_mp, W_mp)

for i in range(M+5):
     C2[n, i] = G[i]

U[0,n]=(C2[n,0]+C2[n,1]*26+C2[n,2]*66+C2[n,3]*26+C2[n,4])/120
U[1,n]=(C2[n,0]+C2[n,1]*237+C2[n,2]*1682+C2[n,3]*1682+C2[n,4]*237+C2[n,5])/3840

for i in range(2,M+1):
    U[i,n]=(C2[n,i-3+2]+C2[n,i-2+2]*26+C2[n,i-1+2]*66+C2[n,i+2]*26+C2[n,i+1+2])/120

U[M+1,n]=(C2[n,M-3+2]+237*C2[n,M-2+2]+1682*C2[n,M-1+2]+1682*C2[n,M+2]+237*C2[n,M+1+2]+C2[n,M+2+2])/3840
U[M+2,n]=(C2[n,M-2+2]+C2[n,M-1+2]*26+C2[n,M+2]*66+C2[n,M+1+2]*26+C2[n,M+2+2])/120


for n in range(3,N+1):
    # --- D1 matrix ---
    D1 = mp.zeros(N+1, 1)  # ✅ allows mpf/mpc objects
    two=mp.mpf(2)
    j=n
    D1[0,0] = -two * Del(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1[1,0] = (
        -two * Del(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat(2, j) / (Tau[2] + Tau[3])
    )

    if j>3:
        for k in range(2, j - 1):
            D1[k,0] = (
                -two * Del(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
                + two * Del(k, j) / (Tau[k] * Tau[k + 1])
                - two * Del(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
                - gamma_mat(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
                + gamma_mat(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
            )

    D1[j - 1,0] = (
        -two * Del(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1[j,0] = (
        two * Del(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    # --- D1 matrix ---
    D1_s = mp.zeros(N+1,1)
    two=mp.mpf(2)
    j=n
    D1_s[0,0] = -two * Del_s(1, j) / mp.mpf(Tau[1] * (Tau[1] + Tau[2])) + gamma_mat_s(1, j) / (Tau[1] + Tau[2])
    #D1[0,1] this second 1 is for mp.mpf only
    D1_s[1,0] = (
        -two * Del_s(2, j) / mp.mpf(Tau[2] * mp.mpf(Tau[2] + Tau[3]))
        + two * Del_s(1, j) / mp.mpf(Tau[1] * Tau[2])
        + gamma_mat_s(2, j) / (Tau[2] + Tau[3])
    )
    for k in range(2, j - 1):
        D1_s[k,0] = (
            -two * Del_s(k - 1, j) / (Tau[k] * (Tau[k - 1] + Tau[k]))
            + two * Del_s(k, j) / (Tau[k] * Tau[k + 1])
            - two * Del_s(k + 1, j) / mp.mpf(Tau[k + 1] * (Tau[k + 1] + Tau[k + 2]))
            - gamma_mat_s(k - 1, j) / mp.mpf(Tau[k - 1] + Tau[k])
            + gamma_mat_s(k + 1, j) / (Tau[k + 1] + Tau[k + 2])
        )
    D1_s[j - 1,0] = (
        -two * Del_s(j - 2, j) / mp.mpf(Tau[j - 1] * (Tau[j - 2] + Tau[j - 1]))
        + two * Del_s(j - 1, j) / mp.mpf(Tau[j - 1] * Tau[j])
        - gamma_mat_s(j - 2, j) / mp.mpf(Tau[j - 2] + Tau[j - 1])
    )
    D1_s[j,0] = (
        two * Del_s(j - 1, j) / mp.mpf(Tau[j] * (Tau[j - 1] + Tau[j]))
        + gamma_mat_s(j - 1, j) / mp.mpf(Tau[j - 1] + Tau[j])
    )

    q1=((d(n,n)+d_s(n,n))-c_)/120
    q2=((d(n,n)+d_s(n,n))-c_)/3840

    # B[0,0]=A1[0,0]         # first Boundary condition
    # B[0,1]=A1[0,1]
    # B[0,2]=A1[0,2]
    # B[0,3]=A1[0,3]
    # B[0,4]=A1[0,4]

    # for i in range(M-2,M+3):
    #     B[M+4,i+2]=B[0,i-(M-2)]

    B[1,0]=A1[1,0]+q1      # Equation with node x_0  # y-y"(0)
    B[1,1]=A1[1,1]+26*q1
    B[1,2]=A1[1,2]+66*q1
    B[1,3]=A1[1,3]+26*q1
    B[1,4]=A1[1,4]+q1
    # B[1,5]=A1[1,5]
    # B[1,6]=A1[1,6]
    # B[1,7]=A1[1,7]
    # B[1,8]=A1[1,8]
    # B[1,9]=A1[1,9]

    for i in range(-7+M,M+3):
        B[M+3,i+2]=B[1,9-(i-(-7+M))]

    B[2,0]=A1[2,0]+q2         # Tau1
    B[2,1]=A1[2,1]+237*q2
    B[2,2]=A1[2,2]+1682*q2
    B[2,3]=A1[2,3]+1682*q2
    B[2,4]=A1[2,4]+237*q2
    B[2,5]=A1[2,5]+q2
    # B[2,6]=A1[2,6]
    # B[2,7]=A1[2,7]
    # B[2,8]=A1[2,8]
    # B[2,9]=A1[2,9]

    for i in range(-7+M,M+3):
        B[M+2,i+2]=B[2,9-(i-(-7+M))]

    # B[3,0]=A1[3,0]                                 # y-y"(1)
    B[3,1]=A1[3,1]+q1
    B[3,2]=A1[3,2]+26*q1
    B[3,3]=A1[3,3]+66*q1
    B[3,4]=A1[3,4]+26*q1
    B[3,5]=A1[3,5]+q1
    # B[3,6]=A1[3,6]
    # B[3,7]=A1[3,7]
    # B[3,8]=A1[3,8]
    # B[3,9]=A1[3,9]

    for i in range(-7+M,M+3):
        B[M+1,i+2]=B[3,9-(i-(-7+M))]

    for i in range(2,M-1):                    # y-y"(i)  i from 2 to M-2
        # B[i+2,-4+i+2]=A1[i+2,-4+i+2]
        # B[i+2,-3+i+2]=A1[i+2,-3+i+2]
        B[i+2,-2+i+2]=A1[i+2,-2+i+2]+q1
        B[i+2,-1+i+2]=A1[i+2,-1+i+2]+26*q1
        B[i+2,i+2]=A1[i+2,i+2]+66*q1
        B[i+2,1+i+2]=A1[i+2,1+i+2]+26*q1
        B[i+2,2+i+2]=A1[i+2,2+i+2]+q1
        # B[i+2,3+i+2]=A1[i+2,3+i+2]
        # B[i+2,4+i+2]=A1[i+2,4+i+2]

    W[0]=t[n]**2
    for i in range(0,M+3):
        W[i+1]=(Xi[0,n]+Xi_s[0,n])*u[i,0]+mp.fdot(Xi[1:N1,n]-Xi[0:N1-1,n]+Xi_s[1:N1,n]-Xi_s[0:N1-1,n],u[i,1:N1])-(Xi[N1-1,n]+Xi_s[N1-1,n])*u[i,N1]+mp.fdot((D1[0:n,0]+D1_s[0:n,0]),U[i,0:n])+f(X[i],t[n])
    W[M+4]=(t[n]**2)*mp.exp(1)

    # ab = extract_banded_form(B, lower_bw=8, upper_bw=8)
    # G = solve_banded((8, 8), ab, W)

    # for i in range(0,M+5):
    #     C[n,i]=G.item(i)

    B_mp = mp.matrix(B)
    W_mp = mp.matrix(W)

    G = mp.lu_solve(B_mp, W_mp)

    for i in range(M+5):
         C[n, i] = G[i]

    U[0,n]=(C[n,0]+C[n,1]*26+C[n,2]*66+C[n,3]*26+C[n,4])/120
    U[1,n]=(C[n,0]+C[n,1]*237+C[n,2]*1682+C[n,3]*1682+C[n,4]*237+C[n,5])/3840

    for i in range(2,M+1):
        U[i,n]=(C[n,i-3+2]+C[n,i-2+2]*26+C[n,i-1+2]*66+C[n,i+2]*26+C[n,i+1+2])/120

    U[M+1,n]=(C[n,M-3+2]+237*C[n,M-2+2]+1682*C[n,M-1+2]+1682*C[n,M+2]+237*C[n,M+1+2]+C[n,M+2+2])/3840
    U[M+2,n]=(C[n,M-2+2]+C[n,M-1+2]*26+C[n,M+2]*66+C[n,M+1+2]*26+C[n,M+2+2])/120

# u_exact=np.zeros((M+3,N1+1))
# u_error=np.zeros((M+3,N1+1))
# for j in range(0,N1+1):
#     for i in range(0,M+3):
#         u_exact[i,j]=(tF[j]**2)*(mp.sin(2*mp.pi*X[i]))
#         u_error[i,j]=abs(u_exact[i,j]-u[i,j])

# U_exact=np.zeros((M+3,N+1))
# U_error=np.zeros((M+3,N+1))
U_exact = mp.matrix(M+3, N+1)
U_error = mp.matrix(M+3, N+1)

for i in range(0,M+3):
    for j in range(0,N+1):
        U_exact[i,j]=(t[j]**2)*(mp.exp(X[i]))
        U_error[i,j]=abs(U_exact[i,j]-U[i,j])


#print(np.max(U_error))
max_error = max(abs(U_error[i,j]) for i in range(U_error.rows) for j in range(U_error.cols))
print(max_error)

# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Meshgrid (coarse grid)
# # --------------------------------------------------
# Xplot_coarse, Yplot_coarse = np.meshgrid(X, t)

# Xplot_coarse = Xplot_coarse.astype(float)
# Yplot_coarse = Yplot_coarse.astype(float)
# V_float1 = np.array(U_exact, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float1.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Exact solution")
# ax.grid(False)

# # Fixed good viewing angle (for paper)
# ax.view_init(elev=28, azim=225)


# # --------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("exact_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("exact_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -------------------------------------------------------------------------------------------------------------------------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# # --------------------------------------------------
# # Convert numerical solution to float
# # --------------------------------------------------
# V_float = np.array(U, dtype=float)

# # --------------------------------------------------
# # Create figure and 3D axis
# # --------------------------------------------------
# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# # --------------------------------------------------
# # Surface plot
# # --------------------------------------------------
# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, V_float.T,
#     cmap='viridis',
#     edgecolor='none',
#     antialiased=True
# )

# # --------------------------------------------------
# # Labels and title
# # --------------------------------------------------
# ax.set_xlabel(r"Space $x$")
# ax.set_ylabel(r"Time $t$")
# ax.set_zlabel(r"Numerical solution ")
# ax.grid(False)

# # Fixed good viewing angle (match exact solution plot)
# ax.view_init(elev=28, azim=225)

# # -------------------------------------------------
# # Layout and save
# # --------------------------------------------------
# plt.tight_layout()

# plt.savefig("numerical_solution_surface_alpha06.png", dpi=300, bbox_inches="tight")
# plt.savefig("numerical_solution_surface_alpha06.pdf", bbox_inches="tight")

# plt.show()
# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)


# # -----------------------------
# # Global plotting style (paper quality)
# # -----------------------------
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "lines.linewidth": 2,
#     "figure.dpi": 120
# })

# #-------------------------------------------------------------------------------------------------------------------------------------

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120
# })

# U_error_float = np.array(U_error, dtype=float)

# fig = plt.figure(figsize=(7.5, 5.5))
# ax = fig.add_subplot(111, projection='3d')

# surf = ax.plot_surface(
#     Xplot_coarse, Yplot_coarse, U_error_float.T,
#     cmap='viridis',
#     linewidth=0,
#     antialiased=True,
#     shade=True
# )

# surf.set_rasterized(True)

# ax.set_xlabel(r"Space $x$", labelpad=10)
# ax.set_ylabel(r"Time $t$", labelpad=10)
# ax.set_zlabel(r"Absolute error", labelpad=10)

# ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# ax.ticklabel_format(axis='z', style='sci', scilimits=(0,0))

# ax.view_init(elev=28, azim=225)
# ax.dist = 11
# ax.grid(False)

# plt.tight_layout()
# plt.savefig("Error.png", dpi=300, bbox_inches="tight")
# plt.savefig("Error.pdf", bbox_inches="tight")
# plt.show()

# for angle in range(0, 360, 5):
#     ax.view_init(elev=30, azim=angle)
#     plt.draw()
#     plt.pause(0.1)

# end=datetime.now()
# print(end-start)

# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.ticker import ScalarFormatter
# from datetime import datetime
# import mpmath as mp

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 12,
#     "axes.labelsize": 13,
#     "axes.titlesize": 13,
#     "legend.fontsize": 11,
#     "xtick.labelsize": 11,
#     "ytick.labelsize": 11,
#     "figure.dpi": 120,
# })

# # --------------------------------------------------
# # mpmath -> numpy float conversion
# # --------------------------------------------------
# def to_float(a, vector=False):
#     """Convert mp.matrix / list of mpf / nested lists to a float ndarray."""
#     if isinstance(a, mp.matrix):
#         a = a.tolist()
#     arr = np.array([[float(v) for v in row] for row in a]) \
#         if (len(a) and isinstance(a[0], (list, tuple))) \
#         else np.array([float(v) for v in a])
#     return arr.ravel() if vector else arr

# X_f = to_float(X, vector=True)
# t_f = to_float(t, vector=True)

# Xplot_coarse, Yplot_coarse = np.meshgrid(X_f, t_f)   # shape (len(t), len(X))

# U_exact_f = to_float(U_exact)   # expected shape (len(X), len(t))
# U_f       = to_float(U)
# U_err_f   = to_float(U_error)

# # --------------------------------------------------
# # Generic surface plot
# # --------------------------------------------------
# def surface(Z, zlabel, fname, sci=False, animate=False):
#     fig = plt.figure(figsize=(7.5, 5.5))
#     ax = fig.add_subplot(111, projection="3d")

#     surf = ax.plot_surface(Xplot_coarse, Yplot_coarse, Z.T,
#                            cmap="viridis", linewidth=0,
#                            antialiased=True, shade=True)
#     surf.set_rasterized(True)

#     ax.set_xlabel(r"Space $x$", labelpad=10)
#     ax.set_ylabel(r"Time $t$", labelpad=10)
#     ax.set_zlabel(zlabel, labelpad=10)
#     if sci:
#         ax.zaxis.set_major_formatter(ScalarFormatter(useMathText=True))
#         ax.ticklabel_format(axis="z", style="sci", scilimits=(0, 0))

#     ax.view_init(elev=28, azim=225)
#     ax.set_box_aspect(None, zoom=0.9)   # replaces deprecated ax.dist
#     ax.grid(False)

#     plt.tight_layout()
#     plt.savefig(f"{fname}.png", dpi=300, bbox_inches="tight")
#     plt.savefig(f"{fname}.pdf", bbox_inches="tight")
#     plt.show()

#     if animate:
#         for angle in range(0, 360, 5):
#             ax.view_init(elev=30, azim=angle)
#             plt.draw()
#             plt.pause(0.1)

# surface(U_exact_f, r"Exact solution",     "exact_solution_surface_alpha06")
# surface(U_f,       r"Numerical solution", "numerical_solution_surface_alpha06")
# surface(U_err_f,   r"Absolute error",     "Error", sci=True)

# end = datetime.now()
# print(end - start)









Example_5 code